# 08. Stage 1 QCHS Personalized Retrieval — Facial Skincare

This notebook evaluates the locked query-only winner and three matched QCHS extensions on the same 2,288 cases, catalog, and exact 1,000-candidate budget:

| Method | Retrieval composition |
|---|---|
| **Query-only winner** | Candidate identifiers, order, and scores from the locked query-only pool. |
| **Profile Sparse QCHS** | Query-only winner fused with BM25 retrieval from the repeated active query and QCHS profile anchors. |
| **Profile Hybrid QCHS** | Query-only winner fused with dense and BM25 retrieval from the QCHS-expanded query. |
| **Profile Full QCHS** | Query-only winner fused with dense QCHS, sparse QCHS, and profile-safe functional-and-brand graph expansion. |

QCHS selects training-safe prior items by functional alignment with the current query. Brand does not drive prior-item selection. After selection, all profile-safe functional and brand facets of the selected prior items may contribute to the profile. Historical review-derived item signals remain in the item-side retrieval representation but are excluded from the user-profile source. Raw prior review text, ratings, and sentiment are not used. Cold and zero-positive-alignment cases copy the query-only candidate identifiers, ranks, and scores exactly.

The personalized winner is selected using Stage 1 retrieval metrics and recorded in a machine-readable contract. A supplementary diagnostic reports QCHS-active coverage and conditional NDCG@5 uplift without modifying the canonical Stage 1 results.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
%pip install -q sentence-transformers faiss-cpu rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 126.2 MB/s eta 0:00:00


In [3]:
# =========================================================
# Load Embedding Model
# =========================================================

from pathlib import Path
import torch
from sentence_transformers import SentenceTransformer

MODEL_DIR = Path(
    "/content/drive/MyDrive/recommendation_benchmark/"
    "models/all-MiniLM-L6-v2"
)

embedding_device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    str(MODEL_DIR),
    device=embedding_device,
    local_files_only=True,
)

print("Embedding model loaded from Google Drive")
print("Embedding device:", embedding_device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded from Google Drive
Embedding device: cuda


In [4]:
from collections import Counter, defaultdict
from pathlib import Path
import json
import math
import re
import time

import faiss
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

In [5]:
# =========================================================
# Config
# =========================================================
CATEGORY_ID = 'face'
CATEGORY_LABEL = 'Facial Skincare'
PROJECT_ROOT = Path('/content/drive/MyDrive/thesis_recsys/categories/facial_skincare')

QUERY_CACHE_PATH = PROJECT_ROOT / 'outputs/query_cache/face_queries.parquet'
QUERY_CONTRACT_PATH = PROJECT_ROOT / 'outputs/query_summary/face_queries_config.json'
PRIOR_HISTORY_PATH = PROJECT_ROOT / 'data/processed/user_sampling/face_user_prior_review_history_training.parquet'
ITEM_DOCS_PATH = PROJECT_ROOT / 'data/processed/items/face_item_docs.parquet'
ITEM_FACETS_PATH = PROJECT_ROOT / 'data/processed/items/face_items_facets.parquet'
RETRIEVAL_ARTIFACT_MANIFEST_PATH = PROJECT_ROOT / 'data/processed/items/retrieval_artifact_manifest_face.json'
STAGE1_SELECTION_PATH = PROJECT_ROOT / 'outputs/stage1_query_retrieval_selection/stage1_method_selection_face.csv'
STAGE1_MANIFEST_PATH = PROJECT_ROOT / 'outputs/stage1_query_retrieval_selection/stage1_run_manifest_face.json'
STAGE1_WINNER_MANIFEST_PATH = PROJECT_ROOT / f'outputs/stage1_query_retrieval_selection/stage1_query_only_winner_{CATEGORY_ID}.json'

OUTPUT_DIR = PROJECT_ROOT / 'outputs/stage1_personalized_retrieval'
CANDIDATE_LISTS_PATH = OUTPUT_DIR / 'personalized_retrieval_candidate_lists_face.parquet'
PER_QUERY_METRICS_PATH = OUTPUT_DIR / 'personalized_retrieval_per_query_metrics_face.parquet'
PROFILE_DIAGNOSTICS_PATH = OUTPUT_DIR / 'personalized_retrieval_user_profile_diagnostics_face.csv'
MANIFEST_PATH = OUTPUT_DIR / 'personalized_retrieval_manifest_face.json'

METHOD_SELECTION_PATH = OUTPUT_DIR / f"personalized_retrieval_method_selection_{CATEGORY_ID}.csv"
WINNER_MANIFEST_PATH = OUTPUT_DIR / f"personalized_retrieval_winner_{CATEGORY_ID}.json"
WINNER_CONTRACT_VERSION = "stage1_personalized_retrieval_winner_v1"
PERSONALIZED_WINNER_METHOD_OVERRIDE = None

ACTIVE_QUERY_COLUMN = "query"
COMPATIBILITY_QUERY_ALIAS = None
PRIOR_HAS_TARGET_PARENT_ASIN = False
DENSE_TEXT_COLUMN = "dense_text"
SPARSE_TEXT_COLUMN = "sparse_text"
BRAND_TEXT_COLUMN = "brand_facet_text"
PROFILE_SAFE_TEXT_COLUMN = "profile_safe_facet_text"
ITEM_EVIDENCE_SCOPE = "catalog_metadata_functional_facets_and_historical_review_signals"
PROFILE_EVIDENCE_SCOPE = "leakage_safe_pre_target_user_prior_items_with_metadata_functional_and_brand_facets"

EXPECTED_QUERY_ROWS = None
EXPECTED_REGIME_COUNTS = None
REGIME_ORDER = ['cold', 'weak', 'moderate', 'strong']
QUERY_PASSTHROUGH_COLUMNS = ['sampling_bracket']

MAX_RETRIEVAL_K = 1000
EVAL_KS = [1, 5, 10, 100, 300, 500, 700, 1000]
MAX_QCHS_PRIOR_ITEMS = 12
MAX_ANCHOR_PHRASES = 16
MAX_ANCHOR_PHRASES_PER_ROLE = 4
QUERY_REPEAT = 3
RRF_K = 60
PROFILE_SPARSE_WEIGHTS = [1.0, 0.8]
PROFILE_HYBRID_WEIGHTS = [1.0, 0.5, 0.7]
PROFILE_FULL_WEIGHTS = [1.0, 0.4, 0.6, 0.6]
ATTENTION_TEMPERATURE = 0.25
GRAPH_ITEMS_PER_FACET_LIMIT = 5000
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
ITEM_EMBEDDING_BATCH_SIZE = 128
QUERY_EMBEDDING_BATCH_SIZE = 256
RANDOM_SEED = 42

FUNCTIONAL_ROLES = {'category_or_product_type',
 'claim_constraint',
 'form_texture',
 'ingredient_or_composition',
 'need_benefit_concern',
 'sensory',
 'target_context'}
PROFILE_ROLES = set(FUNCTIONAL_ROLES) | {"brand"}
TOKEN_EQUIVALENCE = {'acne': 'acne',
 'blemish': 'acne',
 'blemishes': 'acne',
 'breakout': 'acne',
 'breakouts': 'acne',
 'brighten': 'brightening',
 'brightening': 'brightening',
 'brightness': 'brightening',
 'cleanser': 'cleanser',
 'cleansers': 'cleanser',
 'cleansing': 'cleanser',
 'cream': 'moisturizer',
 'creams': 'moisturizer',
 'dark': 'dark',
 'dry': 'dryness',
 'dryness': 'dryness',
 'hydrate': 'hydration',
 'hydrated': 'hydration',
 'hydrating': 'hydration',
 'hydration': 'hydration',
 'hyperpigmentation': 'dark_spots',
 'lotion': 'moisturizer',
 'lotions': 'moisturizer',
 'mask': 'mask',
 'masks': 'mask',
 'moisturize': 'moisturizer',
 'moisturizer': 'moisturizer',
 'moisturizers': 'moisturizer',
 'moisturizing': 'moisturizer',
 'oiliness': 'oiliness',
 'oily': 'oiliness',
 'pigmentation': 'dark_spots',
 'pore': 'pores',
 'pores': 'pores',
 'sensitive': 'sensitivity',
 'sensitivity': 'sensitivity',
 'serum': 'serum',
 'serums': 'serum',
 'spot': 'dark_spots',
 'spots': 'dark_spots',
 'toner': 'toner',
 'toners': 'toner',
 'treatment': 'treatment',
 'treatments': 'treatment',
 'wash': 'cleanser',
 'wrinkle': 'wrinkles',
 'wrinkles': 'wrinkles'}
LINGUISTIC_STOPWORDS = {'a',
 'an',
 'and',
 'are',
 'as',
 'at',
 'be',
 'by',
 'for',
 'from',
 'in',
 'into',
 'is',
 'it',
 'of',
 'on',
 'or',
 'that',
 'the',
 'this',
 'to',
 'with',
 'without',
 'you',
 'your'}
GENERIC_ANCHOR_TOKENS = {'care',
 'face',
 'facial',
 'item',
 'items',
 'product',
 'products',
 'routine',
 'skin',
 'skincare',
 'solution',
 'solutions',
 'support'}
GENERIC_UTILITY_TOKENS = {'blend',
 'care',
 'complex',
 'formula',
 'natural',
 'product',
 'products',
 'routine',
 'solution',
 'solutions',
 'support',
 'wellness'}
ATTENTION_STOPWORDS = {'a',
 'an',
 'and',
 'are',
 'as',
 'at',
 'be',
 'blend',
 'boost',
 'by',
 'care',
 'complex',
 'face',
 'facial',
 'for',
 'formula',
 'from',
 'help',
 'helps',
 'in',
 'into',
 'is',
 'it',
 'its',
 'of',
 'on',
 'or',
 'over',
 'product',
 'promote',
 'promotes',
 'routine',
 'skin',
 'skincare',
 'solution',
 'support',
 'supports',
 'that',
 'the',
 'these',
 'this',
 'those',
 'to',
 'under',
 'with',
 'without'}

PROFILE_METHODS = [
    "profile_sparse_qchs",
    "profile_hybrid_qchs",
    "profile_full_qchs",
]
PROFILE_METHOD_LABELS = {
    "profile_sparse_qchs": "Profile Sparse QCHS",
    "profile_hybrid_qchs": "Profile Hybrid QCHS",
    "profile_full_qchs": "Profile Full QCHS",
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", QUERY_CACHE_PATH)
print("Input:", PRIOR_HISTORY_PATH)
print("Input winner contract:", STAGE1_WINNER_MANIFEST_PATH)
print("Output:", OUTPUT_DIR)


Input: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/query_cache/face_queries.parquet
Input: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/user_sampling/face_user_prior_review_history_training.parquet
Input winner contract: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_query_retrieval_selection/stage1_query_only_winner_face.json
Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_personalized_retrieval


In [6]:
# =========================================================
# Helpers
# =========================================================
def normalize_space(value):
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\n", " ").replace("\t", " ")).strip()


def load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def require_columns(frame, required, frame_name):
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise RuntimeError(f"{frame_name} missing required columns: {missing}")


def boolean_series(values):
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(values):
        return values.fillna(0).astype(float).ne(0)
    normalized = values.fillna("").astype(str).str.strip().str.lower()
    return normalized.isin({"1", "true", "t", "yes", "y"})


def canonical_tokens(value):
    text = normalize_space(value).lower().replace("_", " ").replace("-", " ").replace("/", " ")
    tokens = re.findall(r"[a-z0-9']+", text)
    return [TOKEN_EQUIVALENCE.get(token, token) for token in tokens]


def tokenize_sparse_document(value):
    return [token for token in canonical_tokens(value) if token not in LINGUISTIC_STOPWORDS]


def tokenize_sparse_query(value):
    stopwords = LINGUISTIC_STOPWORDS | GENERIC_ANCHOR_TOKENS
    return [token for token in canonical_tokens(value) if token not in stopwords]


def stable_unique(values):
    return list(dict.fromkeys(values))


def bm25_top_exact(bm25, query_tokens, top_k):
    scores = np.asarray(bm25.get_scores(query_tokens), dtype=np.float64)
    if len(scores) == 0:
        return np.array([], dtype=np.int64), np.array([], dtype=np.float64)
    if not np.isfinite(scores).all():
        raise RuntimeError("BM25 scores contain non-finite values.")
    k_eff = min(int(top_k), len(scores))
    stable_position = np.arange(len(scores), dtype=np.int64)
    order = np.lexsort((stable_position, -scores))
    selected = order[:k_eff].astype(np.int64)
    return selected, scores[selected]


def top_exact_item_ids(bm25, query_tokens, item_ids, top_k):
    selected, _ = bm25_top_exact(bm25, query_tokens, top_k)
    return [item_ids[int(index)] for index in selected]


class _SyntheticBM25:
    def __init__(self, scores):
        self._scores = np.asarray(scores, dtype=np.float64)

    def get_scores(self, query_tokens):
        return self._scores.copy()


def validate_bm25_top_exact_helper():
    cases = [
        ("zero_positive", [0.0, 0.0, 0.0, 0.0], 3),
        ("fewer_than_k_positive", [3.0, 1.0, 0.0, 0.0], 3),
        ("exactly_k_positive", [3.0, 2.0, 1.0, 0.0], 3),
        ("more_than_k_positive", [4.0, 3.0, 2.0, 1.0], 3),
        ("tied_zero_scores", [0.0, 0.0, 0.0, 0.0, 0.0], 4),
        ("negative_scores", [1.0, 0.0, -0.5, -0.5, -1.0], 5),
        ("catalog_smaller_than_k", [2.0, 0.0], 5),
    ]
    for name, scores, top_k in cases:
        bm25 = _SyntheticBM25(scores)
        first_idx, first_scores = bm25_top_exact(bm25, ["query"], top_k)
        second_idx, second_scores = bm25_top_exact(bm25, ["query"], top_k)
        expected_k = min(top_k, len(scores))
        if len(first_idx) != expected_k:
            raise RuntimeError(f"BM25 exact-K helper failed candidate count test: {name}")
        if not np.array_equal(first_idx, second_idx) or not np.array_equal(first_scores, second_scores):
            raise RuntimeError(f"BM25 exact-K helper is not deterministic: {name}")
        expected_order = np.lexsort((np.arange(len(scores), dtype=np.int64), -np.asarray(scores, dtype=np.float64)))[:expected_k]
        if not np.array_equal(first_idx, expected_order):
            raise RuntimeError(f"BM25 exact-K helper ordering mismatch: {name}")


validate_bm25_top_exact_helper()


def top_items_from_score_map(score_map, top_k):
    return [
        item_id
        for item_id, _ in sorted(score_map.items(), key=lambda value: (-value[1], value[0]))[:top_k]
    ]


def rrf_fuse(source_lists, weights, top_k):
    scores = defaultdict(float)
    for source_items, weight in zip(source_lists, weights):
        for rank, item_id in enumerate(source_items, start=1):
            scores[item_id] += float(weight) / (RRF_K + rank)
    ranked = sorted(scores, key=lambda item_id: (-scores[item_id], item_id))[:top_k]
    return ranked, [float(scores[item_id]) for item_id in ranked]


def rank_metrics(rank, k):
    hit = int(0 < rank <= k)
    ndcg = 1.0 / math.log2(rank + 1) if hit else 0.0
    mrr = 1.0 / rank if hit else 0.0
    return hit, ndcg, mrr


def weighted_overlap(query_terms_value, item_terms, token_idf):
    query_term_set = set(query_terms_value)
    item_term_set = set(item_terms)
    if not query_term_set:
        return 0.0
    denominator = sum(token_idf.get(token, 1.0) for token in query_term_set)
    numerator = sum(token_idf.get(token, 1.0) for token in query_term_set & item_term_set)
    return float(numerator / max(denominator, 1e-12))


def softmax_weights(scores):
    values = np.asarray(scores, dtype=float)
    if len(values) == 0:
        return np.array([], dtype=float)
    shifted = values / ATTENTION_TEMPERATURE
    shifted -= shifted.max()
    weights = np.exp(shifted)
    return weights / weights.sum()


def normalized_entropy(weights):
    values = np.asarray(weights, dtype=float)
    values = values[values > 0]
    if len(values) <= 1:
        return 0.0
    entropy = -float(np.sum(values * np.log(values)))
    return entropy / math.log(len(values))


def prior_depth_bin(count):
    count = int(count)
    if count == 0:
        return "0"
    if count <= 2:
        return "1-2"
    if count <= 4:
        return "3-4"
    if count <= 9:
        return "5-9"
    return "10+"


def contains_token_sequence(sequence, subsequence):
    if not subsequence or len(subsequence) > len(sequence):
        return False
    width = len(subsequence)
    return any(tuple(sequence[start:start + width]) == tuple(subsequence) for start in range(len(sequence) - width + 1))


def query_terms(text):
    return stable_unique(
        token for token in canonical_tokens(text)
        if token not in ATTENTION_STOPWORDS and len(token) > 1
    )


In [7]:
# =========================================================
# Load Inputs and Validate Contracts
# =========================================================
required_paths = [
    QUERY_CACHE_PATH,
    QUERY_CONTRACT_PATH,
    PRIOR_HISTORY_PATH,
    ITEM_DOCS_PATH,
    ITEM_FACETS_PATH,
    RETRIEVAL_ARTIFACT_MANIFEST_PATH,
    STAGE1_SELECTION_PATH,
    STAGE1_MANIFEST_PATH,
    STAGE1_WINNER_MANIFEST_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing required inputs: {missing_paths}")

query_contract = load_json(QUERY_CONTRACT_PATH)
retrieval_manifest = load_json(RETRIEVAL_ARTIFACT_MANIFEST_PATH)
stage1_manifest = load_json(STAGE1_MANIFEST_PATH)
stage1_winner = load_json(STAGE1_WINNER_MANIFEST_PATH)
stage1_selection = pd.read_csv(STAGE1_SELECTION_PATH)

if query_contract.get("active_query_column") != ACTIVE_QUERY_COLUMN:
    raise RuntimeError("Notebook 06 active query column must be query.")
if query_contract.get("evidence_scope") != "target_review_safe_signals_only":
    raise RuntimeError("Notebook 06 query evidence scope mismatch.")
if query_contract.get("user_prior_evidence_used") is not False:
    raise RuntimeError("Notebook 06 must not use user-prior evidence.")
if query_contract.get("historical_review_evidence_used") is not False:
    raise RuntimeError("Notebook 06 must not use historical-review evidence.")

if retrieval_manifest.get("evidence_scope") != ITEM_EVIDENCE_SCOPE:
    raise RuntimeError("Notebook 04 evidence scope mismatch.")
if retrieval_manifest.get("dense_source") != DENSE_TEXT_COLUMN:
    raise RuntimeError("Notebook 04 dense source must be dense_text.")
if retrieval_manifest.get("sparse_source") != SPARSE_TEXT_COLUMN:
    raise RuntimeError("Notebook 04 sparse source must be sparse_text.")
if retrieval_manifest.get("historical_review_reputation_enabled") is not True:
    raise RuntimeError("Historical review-derived item signals must be enabled.")
if retrieval_manifest.get("review_reputation_graph_enabled") is not True:
    raise RuntimeError("Historical review-derived graph must be enabled.")
if retrieval_manifest.get("brand_in_functional_graph") is not False:
    raise RuntimeError("Brand must remain separate from product-functional facets.")
if retrieval_manifest.get("brand_graph_enabled") is not True:
    raise RuntimeError("Notebook 04 must preserve brand graph edges.")
if retrieval_manifest.get("brand_in_retrieval_text") is not True:
    raise RuntimeError("Notebook 04 must retain brand in production retrieval text.")
if retrieval_manifest.get("brand_in_profile_source_text") is not True:
    raise RuntimeError("Notebook 04 must retain brand in the profile-safe source.")
if retrieval_manifest.get("brand_in_synthetic_query") is not False:
    raise RuntimeError("Brand must remain excluded from the synthetic query.")

if stage1_manifest.get("evidence_scope") != ITEM_EVIDENCE_SCOPE:
    raise RuntimeError("Notebook 07 evidence scope mismatch.")
if stage1_manifest.get("query_evidence_scope") != "target_review_safe_signals_only":
    raise RuntimeError("Notebook 07 query evidence scope mismatch.")
if stage1_manifest.get("user_prior_enabled") is not False:
    raise RuntimeError("Notebook 07 baseline must not use user prior.")
if stage1_manifest.get("historical_review_reputation_enabled") is not True:
    raise RuntimeError("Notebook 07 baseline must use historical review-derived item signals.")
if stage1_manifest.get("review_reputation_graph_enabled") is not True:
    raise RuntimeError("Notebook 07 baseline must enable the historical review-derived graph.")
if stage1_manifest.get("brand_graph_enabled") is not True:
    raise RuntimeError("Notebook 07 must preserve brand as a catalog graph channel.")
if stage1_manifest.get("brand_in_retrieval_text") is not True:
    raise RuntimeError("Notebook 07 must retain brand in the item retrieval representation.")
if stage1_manifest.get("brand_in_candidate_output") is not True:
    raise RuntimeError("Notebook 07 candidate output must retain brand.")
if stage1_manifest.get("brand_query_matching_enabled") is not False:
    raise RuntimeError("Notebook 07 must not use synthetic-query brand matching.")
if stage1_manifest.get("candidate_pool_depth") != MAX_RETRIEVAL_K:
    raise RuntimeError("Notebook 07 candidate depth mismatch.")
if stage1_manifest.get("candidate_budget_policy") != "exact_k_all_methods":
    raise RuntimeError("Notebook 07 must use exact-K candidate budgets for all methods.")
if stage1_manifest.get("candidate_budget_k") != MAX_RETRIEVAL_K:
    raise RuntimeError("Notebook 07 exact-K budget mismatch.")
if stage1_manifest.get("exact_k_validation_passed") is not True:
    raise RuntimeError("Notebook 07 exact-K validation did not pass.")
if stage1_manifest.get("embedding_model") != EMBEDDING_MODEL_NAME:
    raise RuntimeError("Notebook 07 embedding model mismatch.")

if stage1_winner.get("contract_version") != "stage1_query_only_winner_v1":
    raise RuntimeError("Notebook 07 winner contract version mismatch.")
if stage1_winner.get("category_id") != CATEGORY_ID:
    raise RuntimeError("Notebook 07 winner contract category mismatch.")
if stage1_winner.get("candidate_budget_policy") != "exact_k_all_methods":
    raise RuntimeError("Notebook 07 winner contract must use exact-K candidate budgets.")
if stage1_winner.get("candidate_budget_k") != MAX_RETRIEVAL_K:
    raise RuntimeError("Notebook 07 winner contract candidate budget mismatch.")
if stage1_winner.get("exact_k_validation_passed") is not True:
    raise RuntimeError("Notebook 07 winner contract did not pass exact-K validation.")
if stage1_winner.get("user_prior_enabled") is not False:
    raise RuntimeError("Notebook 07 winner must remain query-only.")
if stage1_winner.get("retrieval_evidence_scope") != ITEM_EVIDENCE_SCOPE:
    raise RuntimeError("Notebook 07 winner retrieval evidence scope mismatch.")
if stage1_winner.get("brand_graph_enabled") is not True:
    raise RuntimeError("Notebook 07 winner contract must preserve brand graph edges.")
if stage1_winner.get("brand_in_retrieval_text") is not True:
    raise RuntimeError("Notebook 07 winner contract must retain brand in item text.")
if stage1_winner.get("brand_in_candidate_output") is not True:
    raise RuntimeError("Notebook 07 winner contract must retain candidate brand.")
if stage1_winner.get("brand_query_matching_enabled") is not False:
    raise RuntimeError("Notebook 07 winner contract must keep query-side brand matching disabled.")

BASELINE_METHOD_SLUG = normalize_space(stage1_winner.get("winner_method_key"))
BASELINE_METHOD_LABEL = normalize_space(stage1_winner.get("winner_method_label"))
BASELINE_CANDIDATES_PATH = Path(normalize_space(stage1_winner.get("winner_candidate_path")))
if not BASELINE_METHOD_SLUG or not BASELINE_METHOD_LABEL:
    raise RuntimeError("Notebook 07 winner contract has an empty method key or label.")
if BASELINE_METHOD_SLUG in PROFILE_METHODS:
    raise RuntimeError("The query-only winner method key conflicts with a personalized method key.")
if not BASELINE_CANDIDATES_PATH.exists():
    raise FileNotFoundError(f"Notebook 07 winner candidate cache is missing: {BASELINE_CANDIDATES_PATH}")

METHOD_ORDER = [BASELINE_METHOD_SLUG, *PROFILE_METHODS]
METHOD_LABELS = {BASELINE_METHOD_SLUG: BASELINE_METHOD_LABEL, **PROFILE_METHOD_LABELS}

require_columns(
    stage1_selection,
    ["selection_rank", "method_key", "retrieval_method", "is_selected_winner"],
    "Notebook 07 selection table",
)
selected_rows = stage1_selection.loc[boolean_series(stage1_selection["is_selected_winner"])].copy()
if len(selected_rows) != 1:
    raise RuntimeError("Notebook 07 selection table must contain exactly one selected winner.")
selected_row = selected_rows.iloc[0]
if normalize_space(selected_row["method_key"]) != BASELINE_METHOD_SLUG:
    raise RuntimeError("Notebook 07 selection table and winner contract method keys differ.")
if normalize_space(selected_row["retrieval_method"]) != BASELINE_METHOD_LABEL:
    raise RuntimeError("Notebook 07 selection table and winner contract method labels differ.")
if stage1_manifest.get("winner_method_key") != BASELINE_METHOD_SLUG:
    raise RuntimeError("Notebook 07 run manifest and winner contract method keys differ.")
if stage1_manifest.get("winner_method_label") != BASELINE_METHOD_LABEL:
    raise RuntimeError("Notebook 07 run manifest and winner contract method labels differ.")
manifest_candidate_path = stage1_manifest.get("candidate_paths", {}).get(BASELINE_METHOD_SLUG)
if not manifest_candidate_path or Path(manifest_candidate_path) != BASELINE_CANDIDATES_PATH:
    raise RuntimeError("Notebook 07 candidate path lineage does not match the winner contract.")

query_schema = pq.ParquetFile(QUERY_CACHE_PATH).schema.names
query_aliases = sorted({"query_C"}.intersection(query_schema))
expected_aliases = [] if COMPATIBILITY_QUERY_ALIAS is None else [COMPATIBILITY_QUERY_ALIAS]
if query_aliases != expected_aliases:
    raise RuntimeError(f"Active query aliases are ambiguous: expected {expected_aliases}, found {query_aliases}.")

required_query_columns = [
    "case_id",
    "user_id",
    "target_parent_asin",
    "regime",
    "target_timestamp_ms",
    ACTIVE_QUERY_COLUMN,
    "query_evidence_source",
    "target_metadata_fallback_used",
    "item_context_fallback_used",
    "historical_review_evidence_used",
    "user_prior_evidence_used",
    "insufficient_review_evidence",
    "query_clean_is_active",
    *QUERY_PASSTHROUGH_COLUMNS,
]
if COMPATIBILITY_QUERY_ALIAS is not None:
    required_query_columns.append(COMPATIBILITY_QUERY_ALIAS)
queries = pd.read_parquet(QUERY_CACHE_PATH, columns=required_query_columns).copy()
require_columns(queries, required_query_columns, "query cache")

for column in ["case_id", "user_id", "target_parent_asin", "regime", ACTIVE_QUERY_COLUMN]:
    queries[column] = queries[column].fillna("").astype(str).map(normalize_space)
queries["target_timestamp_ms"] = pd.to_numeric(queries["target_timestamp_ms"], errors="raise").astype("int64")
queries["active_query_text"] = queries[ACTIVE_QUERY_COLUMN]

observed_query_regime_counts = queries["regime"].value_counts().reindex(REGIME_ORDER, fill_value=0).astype(int).to_dict()
if len(set(observed_query_regime_counts.values())) != 1:
    raise RuntimeError(f"Query cache regime counts must be balanced: {observed_query_regime_counts}")
contract_target_per_regime = int(query_contract.get("target_per_regime", -1))
expected_query_regime_counts = {regime: int(contract_target_per_regime) for regime in REGIME_ORDER}
if observed_query_regime_counts != expected_query_regime_counts:
    raise RuntimeError(
        f"Query cache counts differ from Notebook 06 contract: actual {observed_query_regime_counts}, "
        f"expected {expected_query_regime_counts}"
    )
EXPECTED_REGIME_COUNTS = dict(observed_query_regime_counts)
EXPECTED_QUERY_ROWS = int(sum(EXPECTED_REGIME_COUNTS.values()))
if len(queries) != EXPECTED_QUERY_ROWS:
    raise RuntimeError(f"Expected {EXPECTED_QUERY_ROWS} query rows, found {len(queries)}.")
if queries["case_id"].duplicated().any():
    raise RuntimeError("case_id must be unique.")
if queries["user_id"].duplicated().any():
    raise RuntimeError("user_id must be unique.")
if queries[["case_id", "user_id", "target_parent_asin", "regime", "active_query_text"]].eq("").any().any():
    raise RuntimeError("Query cache contains an empty required value.")
if queries["regime"].value_counts().reindex(REGIME_ORDER, fill_value=0).astype(int).to_dict() != EXPECTED_REGIME_COUNTS:
    raise RuntimeError(f"Unexpected regime counts: {queries['regime'].value_counts().to_dict()}")
if not queries["query_evidence_source"].eq("target_review_safe_signals_only").all():
    raise RuntimeError("Every query must use target-review-safe signals only.")
for column in [
    "target_metadata_fallback_used",
    "item_context_fallback_used",
    "historical_review_evidence_used",
    "user_prior_evidence_used",
    "insufficient_review_evidence",
    "query_clean_is_active",
]:
    if boolean_series(queries[column]).any():
        raise RuntimeError(f"Query audit field must be false: {column}")
if COMPATIBILITY_QUERY_ALIAS is not None:
    if not queries[COMPATIBILITY_QUERY_ALIAS].fillna("").astype(str).map(normalize_space).eq(queries[ACTIVE_QUERY_COLUMN]).all():
        raise RuntimeError(f"{COMPATIBILITY_QUERY_ALIAS} must equal query.")

item_docs = pd.read_parquet(
    ITEM_DOCS_PATH,
    columns=["parent_asin", DENSE_TEXT_COLUMN, SPARSE_TEXT_COLUMN, BRAND_TEXT_COLUMN, PROFILE_SAFE_TEXT_COLUMN],
).copy()
item_docs["parent_asin"] = item_docs["parent_asin"].fillna("").astype(str).map(normalize_space)
for column in [DENSE_TEXT_COLUMN, SPARSE_TEXT_COLUMN, BRAND_TEXT_COLUMN, PROFILE_SAFE_TEXT_COLUMN]:
    item_docs[column] = item_docs[column].fillna("").astype(str).map(normalize_space)
if item_docs["parent_asin"].eq("").any() or item_docs["parent_asin"].duplicated().any():
    raise RuntimeError("Item documents contain empty or duplicate parent_asin values.")
if item_docs[DENSE_TEXT_COLUMN].eq("").any():
    raise RuntimeError("dense_text must be non-empty for every item.")
if item_docs[SPARSE_TEXT_COLUMN].eq("").any():
    raise RuntimeError("sparse_text must be non-empty for every item.")
if item_docs[BRAND_TEXT_COLUMN].eq("").all():
    raise RuntimeError("Notebook 04 item docs must retain non-empty brand facets.")
brand_missing_profile = item_docs.apply(
    lambda row: (
        bool(row[BRAND_TEXT_COLUMN])
        and row[BRAND_TEXT_COLUMN].lower() not in row[PROFILE_SAFE_TEXT_COLUMN].lower()
    ),
    axis=1,
)
if brand_missing_profile.any():
    raise RuntimeError("Brand must be present in profile_safe_facet_text for branded items.")
if len(item_docs) == 0:
    raise RuntimeError("Global Review catalog must contain at least one item.")
item_docs = item_docs.sort_values("parent_asin", kind="stable").reset_index(drop=True)
catalog_size = int(len(item_docs))
EXPECTED_CANDIDATE_K = min(int(MAX_RETRIEVAL_K), catalog_size)
if EXPECTED_CANDIDATE_K <= 0:
    raise RuntimeError("Exact-K candidate budget must be positive.")
if stage1_manifest.get("catalog_size") != catalog_size:
    raise RuntimeError("Notebook 07 catalog size does not match the current Global Review catalog.")
if stage1_winner.get("effective_candidate_count_per_query") != EXPECTED_CANDIDATE_K:
    raise RuntimeError("Notebook 07 winner contract effective candidate count mismatch.")

facet_columns = [
    "parent_asin",
    "facet_value_norm",
    "facet_role",
    "is_brand",
    "is_review_derived",
    "is_product_functional_facet",
    "is_query_safe",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
]
facets = pd.read_parquet(ITEM_FACETS_PATH, columns=facet_columns).copy()
require_columns(facets, facet_columns, "item facets")

prior_schema_columns = pq.ParquetFile(PRIOR_HISTORY_PATH).schema.names
forbidden_prior_fragments = (
    "review_text", "review_body", "review_title", "raw_review",
    "brand", "manufacturer", "seller", "rating", "sentiment",
    "prompt", "response", "llm",
)
forbidden_prior_columns = sorted(
    column for column in prior_schema_columns
    if any(fragment in column.lower() for fragment in forbidden_prior_fragments)
)
if forbidden_prior_columns:
    raise RuntimeError(f"Prior-history artifact contains forbidden fields: {forbidden_prior_columns}")

prior_columns = [
    "case_id",
    "user_id",
    "target_timestamp_ms",
    "prior_item_id",
    "prior_timestamp_ms",
]
if PRIOR_HAS_TARGET_PARENT_ASIN:
    prior_columns.append("target_parent_asin")
prior_history = pd.read_parquet(PRIOR_HISTORY_PATH, columns=prior_columns).copy()
require_columns(prior_history, prior_columns, "prior history")

baseline_schema = pq.ParquetFile(BASELINE_CANDIDATES_PATH).schema.names
required_baseline_columns = [
    "case_id",
    "user_id",
    "regime",
    "target_parent_asin",
    "candidate_parent_asin",
    "candidate_rank",
    "candidate_score",
    "method_key",
    "retrieval_method",
    "score_semantics",
    "candidate_brand_facet_text",
    "is_target",
]
missing_baseline_columns = [
    column for column in required_baseline_columns
    if column not in baseline_schema
]
if missing_baseline_columns:
    raise RuntimeError(f"Notebook 07 winner candidates are missing columns: {missing_baseline_columns}")
baseline_candidates = pd.read_parquet(
    BASELINE_CANDIDATES_PATH,
    columns=required_baseline_columns,
).copy()
require_columns(baseline_candidates, required_baseline_columns, "Notebook 07 winner candidates")
for column in [
    "case_id", "user_id", "regime", "target_parent_asin", "candidate_parent_asin",
    "method_key", "retrieval_method", "score_semantics", "candidate_brand_facet_text",
]:
    baseline_candidates[column] = baseline_candidates[column].fillna("").astype(str).map(normalize_space)
baseline_candidates["candidate_rank"] = pd.to_numeric(
    baseline_candidates["candidate_rank"], errors="raise"
).astype(int)
baseline_candidates["candidate_score"] = pd.to_numeric(
    baseline_candidates["candidate_score"], errors="raise"
).astype(float)
baseline_candidates["is_target"] = boolean_series(baseline_candidates["is_target"])
baseline_candidates["candidate_brand_facet_text"] = (
    baseline_candidates["candidate_brand_facet_text"].fillna("").astype(str).map(normalize_space)
)
item_brand_map = item_docs.set_index("parent_asin")[BRAND_TEXT_COLUMN]
expected_candidate_brand = (
    baseline_candidates["candidate_parent_asin"].map(item_brand_map).fillna("").astype(str)
)
if not baseline_candidates["candidate_brand_facet_text"].eq(expected_candidate_brand).all():
    raise RuntimeError("Notebook 07 candidate brand values do not match Notebook 04 item docs.")
if not baseline_candidates["method_key"].eq(BASELINE_METHOD_SLUG).all():
    raise RuntimeError("Notebook 07 winner candidate cache method_key mismatch.")
if not baseline_candidates["retrieval_method"].eq(BASELINE_METHOD_LABEL).all():
    raise RuntimeError("Notebook 07 winner candidate cache retrieval_method mismatch.")
baseline_candidates = baseline_candidates.sort_values(
    ["case_id", "candidate_rank"], kind="stable"
).reset_index(drop=True)

print("Rows: queries", len(queries))
print("Rows: items", len(item_docs))
print("Query-only winner:", BASELINE_METHOD_SLUG, "-", BASELINE_METHOD_LABEL)
print("Rows: baseline candidates", len(baseline_candidates))
print("Validation: input contracts passed")


Rows: queries 2288
Rows: items 77502
Query-only winner: graph_hybrid - Graph-Hybrid
Rows: baseline candidates 2288000
Validation: input contracts passed


In [8]:
# =========================================================
# Validate Prior History and Build Metadata Functional-Facet Index
# =========================================================
query_case_ids = set(queries["case_id"])
query_users = queries.set_index("case_id")["user_id"]
query_targets = queries.set_index("case_id")["target_parent_asin"]
query_timestamps = queries.set_index("case_id")["target_timestamp_ms"]
item_ids = item_docs["parent_asin"].tolist()
item_id_set = set(item_ids)

if set(baseline_candidates["case_id"]) != query_case_ids:
    raise RuntimeError("Notebook 07 winner candidate cases do not match the query cache.")
if baseline_candidates.duplicated(["case_id", "candidate_parent_asin"]).any():
    raise RuntimeError("Query-only winner candidate cache contains within-case duplicates.")
if not baseline_candidates["candidate_parent_asin"].isin(item_id_set).all():
    raise RuntimeError("Query-only winner candidate cache contains an item outside the Global Review catalog.")

baseline_counts = baseline_candidates.groupby("case_id").size()
if not baseline_counts.eq(EXPECTED_CANDIDATE_K).all():
    bad_counts = baseline_counts.loc[~baseline_counts.eq(EXPECTED_CANDIDATE_K)].head(10).to_dict()
    raise RuntimeError(f"Notebook 07 query-only winner baseline violates exact-K candidate counts: {bad_counts}")
for case_id, group in baseline_candidates.groupby("case_id", sort=False):
    ranks = group["candidate_rank"].tolist()
    if ranks != list(range(1, EXPECTED_CANDIDATE_K + 1)):
        raise RuntimeError(f"Query-only winner candidate ranks are not exactly 1..{EXPECTED_CANDIDATE_K} for case {case_id}.")
    scores = group["candidate_score"].to_numpy(dtype=float)
    if not np.isfinite(scores).all() or np.any(np.diff(scores) > 1e-12):
        raise RuntimeError(f"Query-only winner candidate scores are invalid for case {case_id}.")

prior_id_columns = ["case_id", "user_id", "prior_item_id"]
if PRIOR_HAS_TARGET_PARENT_ASIN:
    prior_id_columns.append("target_parent_asin")
for column in prior_id_columns:
    prior_history[column] = prior_history[column].fillna("").astype(str).map(normalize_space)
for column in ["target_timestamp_ms", "prior_timestamp_ms"]:
    prior_history[column] = pd.to_numeric(prior_history[column], errors="raise").astype("int64")

prior_history = prior_history[
    prior_history["case_id"].isin(query_case_ids)
    & prior_history["prior_item_id"].ne("")
].copy()
if len(prior_history):
    if not prior_history["user_id"].eq(prior_history["case_id"].map(query_users)).all():
        raise RuntimeError("Prior-history user_id does not match the query case.")
    if PRIOR_HAS_TARGET_PARENT_ASIN:
        if not prior_history["target_parent_asin"].eq(prior_history["case_id"].map(query_targets)).all():
            raise RuntimeError("Prior-history target item does not match the query case.")
    if not prior_history["target_timestamp_ms"].eq(prior_history["case_id"].map(query_timestamps)).all():
        raise RuntimeError("Prior-history target timestamp does not match the query case.")
    if not prior_history["prior_timestamp_ms"].lt(prior_history["target_timestamp_ms"]).all():
        raise RuntimeError("Prior history contains an interaction at or after the target timestamp.")
    if prior_history["prior_item_id"].eq(prior_history["case_id"].map(query_targets)).any():
        raise RuntimeError("Prior history contains the held-out target item.")

outside_item_universe = prior_history[~prior_history["prior_item_id"].isin(item_id_set)].copy()
outside_item_universe.to_csv(
    OUTPUT_DIR / f"prior_history_items_outside_item_universe_{CATEGORY_ID}.csv",
    index=False,
)
prior_history = prior_history[prior_history["prior_item_id"].isin(item_id_set)].copy()

prior_items = (
    prior_history
    .groupby(["case_id", "user_id", "prior_item_id"], as_index=False)
    .agg(
        prior_review_count=("prior_timestamp_ms", "size"),
        latest_prior_timestamp_ms=("prior_timestamp_ms", "max"),
        target_timestamp_ms=("target_timestamp_ms", "first"),
    )
    .sort_values(
        ["case_id", "latest_prior_timestamp_ms", "prior_item_id"],
        ascending=[True, False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

prior_event_counts = prior_history.groupby("case_id").size()
prior_item_counts = prior_items.groupby("case_id").size()
queries["prior_review_event_count"] = queries["case_id"].map(prior_event_counts).fillna(0).astype(int)
queries["prior_unique_item_count"] = queries["case_id"].map(prior_item_counts).fillna(0).astype(int)
queries["prior_depth_bin"] = queries["prior_unique_item_count"].map(prior_depth_bin)

for column in [
    "is_brand",
    "is_review_derived",
    "is_product_functional_facet",
    "is_query_safe",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
]:
    facets[column] = boolean_series(facets[column])
facets["parent_asin"] = facets["parent_asin"].fillna("").astype(str).map(normalize_space)
facets["facet_role"] = facets["facet_role"].fillna("").astype(str).map(normalize_space).str.lower()
facets["facet_value_norm"] = facets["facet_value_norm"].fillna("").astype(str).map(normalize_space).str.lower()

explicit_metadata_mask = (
    facets["is_product_functional_facet"]
    & facets["is_query_safe"]
    & ~facets["is_brand"]
    & ~facets["is_review_derived"]
    & ~facets["is_generic_category_anchor"]
    & ~facets["is_generic_utility_token"]
    & ~facets["is_context_dependent_utility_token"]
    & facets["is_metadata_facet_source"]
    & ~facets["is_disallowed_nonfacet_source"]
)
explicit_brand_mask = (
    facets["facet_role"].eq("brand")
    & facets["is_brand"]
    & ~facets["is_review_derived"]
    & ~facets["is_generic_category_anchor"]
    & ~facets["is_generic_utility_token"]
    & ~facets["is_context_dependent_utility_token"]
    & facets["is_metadata_facet_source"]
    & ~facets["is_disallowed_nonfacet_source"]
)
explicit_profile_safe_mask = explicit_metadata_mask | explicit_brand_mask

if not facets["is_core_graph_facet"].eq(explicit_metadata_mask).all():
    raise RuntimeError("Notebook 04 is_core_graph_facet does not match the explicit metadata mask.")
if not facets["is_brand_graph_facet"].eq(explicit_brand_mask).all():
    raise RuntimeError("Notebook 04 is_brand_graph_facet does not match the explicit brand mask.")
if not facets["is_profile_safe"].eq(explicit_profile_safe_mask).all():
    raise RuntimeError("Notebook 04 is_profile_safe does not match functional-plus-brand profile policy.")

alignment_facets = facets.loc[
    explicit_metadata_mask
    & facets["parent_asin"].isin(item_id_set)
    & facets["facet_value_norm"].ne("")
].copy()
profile_safe_facets = facets.loc[
    explicit_profile_safe_mask
    & facets["parent_asin"].isin(item_id_set)
    & facets["facet_value_norm"].ne("")
].copy()

if alignment_facets.empty:
    raise RuntimeError("No metadata product-functional facets are available for QCHS prior-item alignment.")
if profile_safe_facets.empty:
    raise RuntimeError("No profile-safe functional or brand facets are available for QCHS expansion.")
if not alignment_facets["facet_role"].isin(FUNCTIONAL_ROLES).all():
    invalid_roles = sorted(
        set(alignment_facets.loc[~alignment_facets["facet_role"].isin(FUNCTIONAL_ROLES), "facet_role"])
    )
    raise RuntimeError(f"Unexpected QCHS alignment facet roles: {invalid_roles}")
if not profile_safe_facets["facet_role"].isin(PROFILE_ROLES).all():
    invalid_roles = sorted(
        set(profile_safe_facets.loc[~profile_safe_facets["facet_role"].isin(PROFILE_ROLES), "facet_role"])
    )
    raise RuntimeError(f"Unexpected profile-safe facet roles: {invalid_roles}")
if alignment_facets["is_brand"].any() or alignment_facets["is_review_derived"].any():
    raise RuntimeError("Brand or review-derived rows entered QCHS prior-item alignment.")
if profile_safe_facets["is_review_derived"].any():
    raise RuntimeError("Historical population-review facets must not enter the user profile.")
for column in [
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_disallowed_nonfacet_source",
]:
    if profile_safe_facets[column].any():
        raise RuntimeError(f"Disallowed rows entered the profile-safe facet channel: {column}")

brand_profile_facets = profile_safe_facets.loc[profile_safe_facets["is_brand"]].copy()
functional_profile_facets = profile_safe_facets.loc[~profile_safe_facets["is_brand"]].copy()
if brand_profile_facets.empty:
    raise RuntimeError("Brand facets must be available in the QCHS profile-safe channel.")
if brand_profile_facets["is_query_safe"].any():
    raise RuntimeError("Brand facets must remain query-unsafe.")
if brand_profile_facets["is_product_functional_facet"].any():
    raise RuntimeError("Brand facets must remain separate from product-functional facets.")
if not brand_profile_facets["is_profile_safe"].all():
    raise RuntimeError("Brand facets must be profile-safe.")

alignment_facets["facet_key"] = (
    alignment_facets["facet_role"] + "::" + alignment_facets["facet_value_norm"]
)
profile_safe_facets["facet_key"] = (
    profile_safe_facets["facet_role"] + "::" + profile_safe_facets["facet_value_norm"]
)
alignment_facets = alignment_facets.drop_duplicates(["parent_asin", "facet_key"]).copy()
profile_safe_facets = profile_safe_facets.drop_duplicates(["parent_asin", "facet_key"]).copy()

items_by_facet = (
    profile_safe_facets
    .sort_values(["facet_key", "parent_asin"], kind="stable")
    .groupby("facet_key", sort=False)["parent_asin"]
    .agg(list)
    .to_dict()
)

profile_facet_document_frequency = (
    profile_safe_facets.groupby("facet_key")["parent_asin"].nunique()
)
profile_facet_idf = {
    key: float(math.log((1.0 + len(item_docs)) / (1.0 + frequency)) + 1.0)
    for key, frequency in profile_facet_document_frequency.items()
}
alignment_facet_document_frequency = (
    alignment_facets.groupby("facet_key")["parent_asin"].nunique()
)
alignment_facet_idf = {
    key: float(math.log((1.0 + len(item_docs)) / (1.0 + frequency)) + 1.0)
    for key, frequency in alignment_facet_document_frequency.items()
}

alignment_item_token_sets = {}
alignment_records_by_item = defaultdict(list)
alignment_token_document_frequency = Counter()
for item_id, group in alignment_facets.groupby("parent_asin", sort=False):
    item_tokens = set()
    for row in group.itertuples(index=False):
        full_phrase_tokens = tuple(canonical_tokens(row.facet_value_norm))
        phrase_tokens = tuple(
            token for token in full_phrase_tokens
            if token not in ATTENTION_STOPWORDS
        )
        if not full_phrase_tokens or not phrase_tokens:
            continue
        item_tokens.update(phrase_tokens)
        alignment_records_by_item[item_id].append({
            "facet_key": row.facet_key,
            "facet_role": row.facet_role,
            "phrase": row.facet_value_norm,
            "phrase_tokens": full_phrase_tokens,
            "tokens": phrase_tokens,
            "idf": alignment_facet_idf[row.facet_key],
            "is_brand": False,
        })
    alignment_item_token_sets[item_id] = item_tokens
    alignment_token_document_frequency.update(item_tokens)

alignment_token_idf = {
    token: float(math.log((1.0 + len(item_docs)) / (1.0 + frequency)) + 1.0)
    for token, frequency in alignment_token_document_frequency.items()
}

facet_records_by_item = defaultdict(list)
for item_id, group in profile_safe_facets.groupby("parent_asin", sort=False):
    for row in group.itertuples(index=False):
        full_phrase_tokens = tuple(canonical_tokens(row.facet_value_norm))
        phrase_tokens = tuple(
            token for token in full_phrase_tokens
            if token not in ATTENTION_STOPWORDS
        )
        if not full_phrase_tokens:
            continue
        facet_records_by_item[item_id].append({
            "facet_key": row.facet_key,
            "facet_role": row.facet_role,
            "phrase": row.facet_value_norm,
            "phrase_tokens": full_phrase_tokens,
            "tokens": phrase_tokens,
            "idf": profile_facet_idf[row.facet_key],
            "is_brand": bool(row.is_brand),
        })

prior_items_by_case = {
    case_id: group[["prior_item_id", "latest_prior_timestamp_ms", "prior_review_count"]].to_dict("records")
    for case_id, group in prior_items.groupby("case_id", sort=False)
}

qchs_alignable_item_ids = {
    item_id for item_id, records in alignment_records_by_item.items() if records
}
profile_safe_item_ids = {
    item_id for item_id, records in facet_records_by_item.items() if records
}
usable_stage1_profile_item_ids = qchs_alignable_item_ids & profile_safe_item_ids

usable_stage1_prior_items = prior_items[
    prior_items["prior_item_id"].isin(usable_stage1_profile_item_ids)
].copy()
profile_safe_prior_items = prior_items[
    prior_items["prior_item_id"].isin(profile_safe_item_ids)
].copy()
usable_stage1_prior_item_counts = usable_stage1_prior_items.groupby("case_id").size()
profile_safe_prior_item_counts = profile_safe_prior_items.groupby("case_id").size()
queries["stage1_usable_prior_item_count"] = (
    queries["case_id"].map(usable_stage1_prior_item_counts).fillna(0).astype(int)
)
queries["stage1_profile_safe_prior_item_count"] = (
    queries["case_id"].map(profile_safe_prior_item_counts).fillna(0).astype(int)
)
queries["stage1_profile_available"] = queries["stage1_usable_prior_item_count"].gt(0)

cold_history_mismatch = queries[
    queries["regime"].eq("cold")
    & (
        queries["prior_review_event_count"].ne(0)
        | queries["prior_unique_item_count"].ne(0)
        | queries["stage1_usable_prior_item_count"].ne(0)
        | queries["stage1_profile_safe_prior_item_count"].ne(0)
    )
].copy()
if len(cold_history_mismatch):
    display(cold_history_mismatch.head(20))
    raise RuntimeError("Cold cases must have zero usable pre-target Stage 1 history.")

non_cold_history_mismatch = queries[
    queries["regime"].ne("cold")
    & queries["prior_unique_item_count"].lt(1)
].copy()
if len(non_cold_history_mismatch):
    display(non_cold_history_mismatch.head(20))
    raise RuntimeError(
        "Every non-cold query must retain at least one training-safe pre-target prior item."
    )

non_cold_zero_qchs_alignable = queries[
    queries["regime"].ne("cold")
    & queries["stage1_usable_prior_item_count"].eq(0)
].copy()
non_cold_zero_profile_safe = queries[
    queries["regime"].ne("cold")
    & queries["stage1_profile_safe_prior_item_count"].eq(0)
].copy()

print("Rows: usable prior events", len(prior_history))
print("Rows: unique prior items", len(prior_items))
print("Rows: QCHS alignment facets", len(alignment_facets))
print("Rows: profile-safe functional facets", len(functional_profile_facets))
print("Rows: profile-safe brand facets", len(brand_profile_facets))
print("Non-cold cases with zero QCHS-alignable prior items:", len(non_cold_zero_qchs_alignable))
print("Non-cold cases with zero profile-safe prior items:", len(non_cold_zero_profile_safe))
print("Validation: prior history and brand-enabled profile facets passed")


Rows: usable prior events 22519
Rows: unique prior items 22144
Rows: QCHS alignment facets 618684
Rows: profile-safe functional facets 618684
Rows: profile-safe brand facets 74751
Non-cold cases with zero QCHS-alignable prior items: 1
Non-cold cases with zero profile-safe prior items: 0
Validation: prior history and brand-enabled profile facets passed


In [9]:
# =========================================================
# QCHS Profile Functions
# =========================================================
def prior_alignment(query_text, item_id):
    query_token_sequence = canonical_tokens(query_text)
    terms = stable_unique(
        token for token in query_token_sequence
        if token not in ATTENTION_STOPWORDS and len(token) > 1
    )
    records = alignment_records_by_item.get(item_id, [])
    if not terms or not records:
        return 0.0

    item_tokens = alignment_item_token_sets.get(item_id, set())
    token_score = weighted_overlap(terms, item_tokens, alignment_token_idf)
    exact_phrase_score = float(
        any(contains_token_sequence(query_token_sequence, record["phrase_tokens"]) for record in records)
    )
    matched_roles = {
        record["facet_role"]
        for record in records
        if set(record["tokens"]) & set(terms)
    }
    role_score = min(len(matched_roles) / 2.0, 1.0)
    return float(0.70 * token_score + 0.20 * exact_phrase_score + 0.10 * role_score)


def selected_facet_keys(selected_items):
    return stable_unique(
        record["facet_key"]
        for item_id in selected_items
        for record in facet_records_by_item.get(item_id, [])
    )


def build_anchors(selected_items, item_weights, query_text):
    # QCHS conditions prior-item selection on the query. Once an item is selected,
    # all of its profile-safe facets, including brand and query-confirming facets, are retained.
    _ = query_text
    anchor_scores = defaultdict(float)
    anchor_support = defaultdict(set)
    anchor_meta = {}

    for item_id, item_weight in zip(selected_items, item_weights):
        for record in facet_records_by_item.get(item_id, []):
            key = record["facet_key"]
            anchor_scores[key] += float(item_weight) * float(record["idf"])
            anchor_support[key].add(item_id)
            anchor_meta[key] = (
                record["facet_role"],
                record["phrase"],
                bool(record["is_brand"]),
            )

    ranked_keys = sorted(
        anchor_scores,
        key=lambda key: (
            -(anchor_scores[key] + 0.05 * math.log1p(len(anchor_support[key]))),
            anchor_meta[key][0],
            anchor_meta[key][1],
        ),
    )

    selected_keys = []
    selected_phrases = []
    selected_roles = []
    role_counts = Counter()
    for key in ranked_keys:
        role, phrase, _ = anchor_meta[key]
        if role_counts[role] >= MAX_ANCHOR_PHRASES_PER_ROLE:
            continue
        selected_keys.append(key)
        selected_phrases.append(phrase)
        selected_roles.append(role)
        role_counts[role] += 1
        if len(selected_keys) >= MAX_ANCHOR_PHRASES:
            break
    return selected_keys, selected_phrases, selected_roles


def empty_profile():
    return {
        "selected_items": [],
        "alignment_scores": [],
        "item_weights": [],
        "selected_facet_keys": [],
        "anchor_keys": [],
        "anchor_phrases": [],
        "anchor_roles": [],
        "attention_entropy": 0.0,
    }


def build_all_prior_profile(query_text, history_items):
    item_ids = [row["prior_item_id"] for row in history_items]
    if not item_ids:
        return empty_profile()
    weights = np.full(len(item_ids), 1.0 / len(item_ids), dtype=float)
    anchor_keys, anchor_phrases, anchor_roles = build_anchors(item_ids, weights, query_text)
    return {
        "selected_items": item_ids,
        "alignment_scores": [0.0] * len(item_ids),
        "item_weights": weights.tolist(),
        "selected_facet_keys": selected_facet_keys(item_ids),
        "anchor_keys": anchor_keys,
        "anchor_phrases": anchor_phrases,
        "anchor_roles": anchor_roles,
        "attention_entropy": normalized_entropy(weights),
    }


def build_qchs_profile(query_text, history_items):
    scored = []
    for row in history_items:
        score = prior_alignment(query_text, row["prior_item_id"])
        if score > 0:
            scored.append((score, int(row["latest_prior_timestamp_ms"]), row["prior_item_id"]))
    scored.sort(key=lambda value: (-value[0], -value[1], value[2]))
    scored = scored[:MAX_QCHS_PRIOR_ITEMS]

    item_ids = [item_id for _, _, item_id in scored]
    scores = [score for score, _, _ in scored]
    if not item_ids:
        return empty_profile()
    weights = softmax_weights(scores)
    anchor_keys, anchor_phrases, anchor_roles = build_anchors(item_ids, weights, query_text)
    return {
        "selected_items": item_ids,
        "alignment_scores": scores,
        "item_weights": weights.tolist(),
        "selected_facet_keys": selected_facet_keys(item_ids),
        "anchor_keys": anchor_keys,
        "anchor_phrases": anchor_phrases,
        "anchor_roles": anchor_roles,
        "attention_entropy": normalized_entropy(weights),
    }


In [10]:
# =========================================================
# Build QCHS Profiles
# =========================================================
profile_by_case = {}
profile_rows = []

for row in queries.itertuples(index=False):
    history = prior_items_by_case.get(row.case_id, [])
    qchs_profile = build_qchs_profile(row.active_query_text, history)
    profile_by_case[row.case_id] = qchs_profile

    scores = qchs_profile["alignment_scores"]
    qchs_selected_prior_item_count = len(qchs_profile["selected_items"])
    qchs_profile_safe_facet_count = len(qchs_profile["selected_facet_keys"])
    qchs_brand_facet_count = sum(
        str(key).startswith("brand::") for key in qchs_profile["selected_facet_keys"]
    )
    qchs_functional_facet_count = sum(
        not str(key).startswith("brand::") for key in qchs_profile["selected_facet_keys"]
    )
    qchs_profile_available = (
        qchs_selected_prior_item_count > 0
        and qchs_profile_safe_facet_count > 0
    )
    profile_fallback_flag = not qchs_profile_available
    profile_fallback_reason = ""
    if profile_fallback_flag:
        profile_fallback_reason = (
            "cold_no_history" if row.regime == "cold" else "no_qchs_aligned_prior"
        )

    profile_rows.append({
        "case_id": row.case_id,
        "user_id": row.user_id,
        "regime": row.regime,
        "target_parent_asin": row.target_parent_asin,
        "raw_prior_item_count": int(row.prior_unique_item_count),
        "training_safe_prior_item_count": int(row.prior_unique_item_count),
        "prior_review_event_count": int(row.prior_review_event_count),
        "prior_unique_item_count": int(row.prior_unique_item_count),
        "prior_depth_bin": row.prior_depth_bin,
        "stage1_usable_prior_item_count": int(row.stage1_usable_prior_item_count),
        "stage1_profile_safe_prior_item_count": int(row.stage1_profile_safe_prior_item_count),
        "stage1_profile_available": bool(row.stage1_profile_available),
        "qchs_selected_prior_item_count": qchs_selected_prior_item_count,
        "qchs_selected_ratio": (
            qchs_selected_prior_item_count / row.prior_unique_item_count
            if row.prior_unique_item_count else 0.0
        ),
        "qchs_alignment_mean": float(np.mean(scores)) if scores else 0.0,
        "qchs_alignment_max": float(np.max(scores)) if scores else 0.0,
        "qchs_attention_entropy": qchs_profile["attention_entropy"],
        "qchs_profile_safe_facet_count": qchs_profile_safe_facet_count,
        "qchs_brand_facet_count": qchs_brand_facet_count,
        "qchs_functional_facet_count": qchs_functional_facet_count,
        "qchs_profile_available": bool(qchs_profile_available),
        "profile_fallback_flag": bool(profile_fallback_flag),
        "profile_fallback_reason": profile_fallback_reason,
        "qchs_selected_facet_count": qchs_profile_safe_facet_count,
        "qchs_selected_brand_facet_count": qchs_brand_facet_count,
        "qchs_selected_functional_facet_count": qchs_functional_facet_count,
        "qchs_anchor_phrase_count": len(qchs_profile["anchor_phrases"]),
        "qchs_anchor_brand_count": sum(
            role == "brand" for role in qchs_profile["anchor_roles"]
        ),
        "qchs_anchor_phrases": " | ".join(qchs_profile["anchor_phrases"]),
        "qchs_selected_prior_items": " | ".join(qchs_profile["selected_items"]),
        "qchs_fallback_flag": int(profile_fallback_flag),
    })

profile_diagnostics = pd.DataFrame(profile_rows)

cold_profile_mismatch = profile_diagnostics[
    profile_diagnostics["regime"].eq("cold")
    & (
        profile_diagnostics["qchs_selected_prior_item_count"].gt(0)
        | profile_diagnostics["qchs_selected_facet_count"].gt(0)
        | profile_diagnostics["qchs_anchor_phrase_count"].gt(0)
    )
].copy()
if len(cold_profile_mismatch):
    display(cold_profile_mismatch.head(20))
    raise RuntimeError("Cold cases must not produce a QCHS profile.")

invalid_fallback_reason_rows = profile_diagnostics[
    profile_diagnostics["profile_fallback_flag"]
    & profile_diagnostics["profile_fallback_reason"].eq("")
].copy()
if len(invalid_fallback_reason_rows):
    display(invalid_fallback_reason_rows.head(20))
    raise RuntimeError("Every QCHS fallback case must have an explicit fallback reason.")

active_fallback_conflict = profile_diagnostics[
    profile_diagnostics["qchs_profile_available"]
    & profile_diagnostics["profile_fallback_flag"]
].copy()
if len(active_fallback_conflict):
    display(active_fallback_conflict.head(20))
    raise RuntimeError("A case cannot be both QCHS-active and a profile fallback.")

non_cold_no_alignment_fallback = profile_diagnostics[
    profile_diagnostics["regime"].ne("cold")
    & profile_diagnostics["profile_fallback_flag"]
].copy()

profile_fallback_by_case = profile_diagnostics.set_index("case_id")["profile_fallback_flag"].astype(bool).to_dict()
profile_fallback_reason_by_case = profile_diagnostics.set_index("case_id")["profile_fallback_reason"].astype(str).to_dict()
qchs_profile_available_by_case = profile_diagnostics.set_index("case_id")["qchs_profile_available"].astype(bool).to_dict()
profile_diagnostics_by_case = profile_diagnostics.set_index("case_id")

print("Rows: profile diagnostics", len(profile_diagnostics))
print("Rows: QCHS-active profiles", int(profile_diagnostics["qchs_profile_available"].sum()))
print("Rows: profile fallbacks", int(profile_diagnostics["profile_fallback_flag"].sum()))
print("Rows: non-cold no-alignment fallbacks", len(non_cold_no_alignment_fallback))
print("Validation: QCHS profile diagnostics passed")

Rows: profile diagnostics 2288
Rows: QCHS-active profiles 1513
Rows: profile fallbacks 775
Rows: non-cold no-alignment fallbacks 203
Validation: QCHS profile diagnostics passed


In [11]:
# =========================================================
# Personalized Retrieval Sources and Fusion
# =========================================================

from tqdm.auto import tqdm


# =========================================================
# Sparse Corpus and BM25 Index
# =========================================================

print("Starting sparse document tokenization")
print("Global Review item documents:", len(item_docs))

sparse_index_start = time.perf_counter()

sparse_corpus_tokens = [
    tokenize_sparse_document(text)
    for text in tqdm(
        item_docs[SPARSE_TEXT_COLUMN],
        total=len(item_docs),
        desc="Sparse document tokenization",
    )
]

if any(len(tokens) == 0 for tokens in sparse_corpus_tokens):
    raise RuntimeError(
        "Every production sparse item document must contain an indexed token."
    )

print("Building BM25 index")

bm25 = BM25Okapi(sparse_corpus_tokens)

offline_bm25_index_runtime_sec = (
    time.perf_counter() - sparse_index_start
)

print(
    "BM25 index runtime:",
    round(offline_bm25_index_runtime_sec, 2),
    "seconds",
)


# =========================================================
# Baseline Candidate Lookup
# =========================================================

baseline_items_by_case = (
    baseline_candidates
    .groupby("case_id", sort=False)["candidate_parent_asin"]
    .agg(list)
    .to_dict()
)

baseline_scores_by_case = (
    baseline_candidates
    .groupby("case_id", sort=False)["candidate_score"]
    .agg(list)
    .to_dict()
)

query_lookup = queries.set_index("case_id")


# =========================================================
# Sparse QCHS Retrieval
# =========================================================

sparse_qchs_items_by_case = {}
graph_qchs_items_by_case = {}
expanded_dense_text_by_case = {}

print("Starting Sparse QCHS retrieval")
print("Queries:", len(queries))

sparse_start = time.perf_counter()

for case_id in tqdm(
    queries["case_id"],
    total=len(queries),
    desc="Sparse QCHS retrieval",
):
    profile = profile_by_case[case_id]

    if not profile["anchor_phrases"]:
        sparse_qchs_items_by_case[case_id] = []
        expanded_dense_text_by_case[case_id] = ""
        continue

    active_query_text = query_lookup.at[
        case_id,
        "active_query_text",
    ]

    active_tokens = tokenize_sparse_query(
        active_query_text
    )

    anchor_tokens = [
        token
        for phrase in profile["anchor_phrases"]
        for token in tokenize_sparse_query(phrase)
    ]

    expanded_tokens = (
        active_tokens * QUERY_REPEAT
        + anchor_tokens
    )

    sparse_qchs_items_by_case[case_id] = (
        top_exact_item_ids(
            bm25,
            expanded_tokens,
            item_ids,
            EXPECTED_CANDIDATE_K,
        )
    )

    expanded_dense_text_by_case[case_id] = (
        normalize_space(
            active_query_text
            + " "
            + " ".join(profile["anchor_phrases"])
        )
    )

sparse_qchs_runtime_sec = (
    time.perf_counter() - sparse_start
)

print(
    "Sparse QCHS runtime:",
    round(sparse_qchs_runtime_sec, 2),
    "seconds",
)


# =========================================================
# Dense QCHS Retrieval
# =========================================================

profile_case_ids = [
    case_id
    for case_id in queries["case_id"]
    if expanded_dense_text_by_case.get(case_id, "")
]

dense_qchs_items_by_case = {
    case_id: []
    for case_id in queries["case_id"]
}

offline_dense_index_runtime_sec = 0.0
dense_qchs_runtime_sec = 0.0

print("Queries requiring Dense QCHS:", len(profile_case_ids))

if profile_case_ids:
    if "model" not in globals():
        raise RuntimeError(
            "Run the embedding-model load cell first."
        )

    print("Starting dense item embedding")
    print("Dense item documents:", len(item_docs))

    dense_index_start = time.perf_counter()

    item_embeddings = model.encode(
        item_docs[DENSE_TEXT_COLUMN].tolist(),
        batch_size=ITEM_EMBEDDING_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype("float32")

    print("Building FAISS dense index")

    dense_index = faiss.IndexFlatIP(
        item_embeddings.shape[1]
    )

    dense_index.add(item_embeddings)

    offline_dense_index_runtime_sec = (
        time.perf_counter() - dense_index_start
    )

    print(
        "Dense item index runtime:",
        round(offline_dense_index_runtime_sec, 2),
        "seconds",
    )

    print("Starting dense query embedding")

    dense_query_start = time.perf_counter()

    dense_query_embeddings = model.encode(
        [
            expanded_dense_text_by_case[case_id]
            for case_id in profile_case_ids
        ],
        batch_size=QUERY_EMBEDDING_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype("float32")

    print("Searching dense QCHS candidates")

    _, dense_positions = dense_index.search(
        dense_query_embeddings,
        EXPECTED_CANDIDATE_K,
    )

    dense_qchs_runtime_sec = (
        time.perf_counter() - dense_query_start
    )

    for row_index, case_id in enumerate(
        tqdm(
            profile_case_ids,
            total=len(profile_case_ids),
            desc="Dense candidate mapping",
        )
    ):
        dense_qchs_items_by_case[case_id] = [
            item_ids[int(position)]
            for position in dense_positions[row_index]
            if int(position) >= 0
        ]

    print(
        "Dense QCHS runtime:",
        round(dense_qchs_runtime_sec, 2),
        "seconds",
    )


# =========================================================
# Graph QCHS Retrieval
# =========================================================

print("Starting Graph QCHS retrieval")

graph_start = time.perf_counter()

for case_id in tqdm(
    queries["case_id"],
    total=len(queries),
    desc="Graph QCHS retrieval",
):
    profile = profile_by_case[case_id]
    graph_scores = defaultdict(float)

    for facet_key in profile["selected_facet_keys"]:
        for item_id in items_by_facet.get(
            facet_key,
            [],
        )[:GRAPH_ITEMS_PER_FACET_LIMIT]:
            graph_scores[item_id] += 1.0

    graph_qchs_items_by_case[case_id] = (
        top_items_from_score_map(
            graph_scores,
            EXPECTED_CANDIDATE_K,
        )
    )

graph_qchs_runtime_sec = (
    time.perf_counter() - graph_start
)

print(
    "Graph QCHS runtime:",
    round(graph_qchs_runtime_sec, 2),
    "seconds",
)


# =========================================================
# Candidate Fusion
# =========================================================

print("Starting RRF fusion")

method_candidate_lists = {
    method: {}
    for method in METHOD_ORDER
}

method_candidate_scores = {
    method: {}
    for method in METHOD_ORDER
}

method_profile_source_used = {
    method: {}
    for method in PROFILE_METHODS
}

fusion_runtime = defaultdict(float)

for case_id in tqdm(
    queries["case_id"],
    total=len(queries),
    desc="RRF fusion",
):
    baseline_items = baseline_items_by_case[case_id]
    baseline_scores = baseline_scores_by_case[case_id]

    sparse_items = sparse_qchs_items_by_case.get(
        case_id,
        [],
    )

    dense_items = dense_qchs_items_by_case.get(
        case_id,
        [],
    )

    graph_items = graph_qchs_items_by_case.get(
        case_id,
        [],
    )

    method_candidate_lists[
        BASELINE_METHOD_SLUG
    ][case_id] = list(baseline_items)

    method_candidate_scores[
        BASELINE_METHOD_SLUG
    ][case_id] = list(baseline_scores)

    method_sources = {
        "profile_sparse_qchs": (
            [
                baseline_items,
                sparse_items,
            ],
            PROFILE_SPARSE_WEIGHTS,
        ),
        "profile_hybrid_qchs": (
            [
                baseline_items,
                dense_items,
                sparse_items,
            ],
            PROFILE_HYBRID_WEIGHTS,
        ),
        "profile_full_qchs": (
            [
                baseline_items,
                dense_items,
                sparse_items,
                graph_items,
            ],
            PROFILE_FULL_WEIGHTS,
        ),
    }

    for method, (
        source_lists,
        weights,
    ) in method_sources.items():
        personalized_source_used = any(
            bool(source)
            for source in source_lists[1:]
        )

        method_profile_source_used[
            method
        ][case_id] = personalized_source_used

        if not personalized_source_used:
            method_candidate_lists[
                method
            ][case_id] = list(baseline_items)

            method_candidate_scores[
                method
            ][case_id] = list(baseline_scores)

            continue

        fusion_start = time.perf_counter()

        ranked, scores = rrf_fuse(
            source_lists,
            weights,
            EXPECTED_CANDIDATE_K,
        )

        fusion_runtime[method] += (
            time.perf_counter() - fusion_start
        )

        method_candidate_lists[
            method
        ][case_id] = ranked

        method_candidate_scores[
            method
        ][case_id] = scores

print("RRF fusion completed")


# =========================================================
# Runtime Components
# =========================================================

method_runtime_components = {
    BASELINE_METHOD_SLUG: {
        "dense_qchs_runtime_sec": 0.0,
        "sparse_qchs_runtime_sec": 0.0,
        "graph_qchs_runtime_sec": 0.0,
        "fusion_runtime_sec": 0.0,
    },
    "profile_sparse_qchs": {
        "dense_qchs_runtime_sec": 0.0,
        "sparse_qchs_runtime_sec": float(
            sparse_qchs_runtime_sec
        ),
        "graph_qchs_runtime_sec": 0.0,
        "fusion_runtime_sec": float(
            fusion_runtime["profile_sparse_qchs"]
        ),
    },
    "profile_hybrid_qchs": {
        "dense_qchs_runtime_sec": float(
            dense_qchs_runtime_sec
        ),
        "sparse_qchs_runtime_sec": float(
            sparse_qchs_runtime_sec
        ),
        "graph_qchs_runtime_sec": 0.0,
        "fusion_runtime_sec": float(
            fusion_runtime["profile_hybrid_qchs"]
        ),
    },
    "profile_full_qchs": {
        "dense_qchs_runtime_sec": float(
            dense_qchs_runtime_sec
        ),
        "sparse_qchs_runtime_sec": float(
            sparse_qchs_runtime_sec
        ),
        "graph_qchs_runtime_sec": float(
            graph_qchs_runtime_sec
        ),
        "fusion_runtime_sec": float(
            fusion_runtime["profile_full_qchs"]
        ),
    },
}


# =========================================================
# Source Contribution
# =========================================================

source_contribution = pd.DataFrame(
    [
        {
            "method_slug": method,
            "method_label": METHOD_LABELS[method],
            "baseline_query_only_winner_used": True,
            "dense_qchs_used": (
                method
                in {
                    "profile_hybrid_qchs",
                    "profile_full_qchs",
                }
            ),
            "sparse_qchs_used": (
                method in PROFILE_METHODS
            ),
            "graph_qchs_used": (
                method == "profile_full_qchs"
            ),
            "queries_with_personalized_source": (
                int(
                    sum(
                        method_profile_source_used[
                            method
                        ].values()
                    )
                )
                if method in PROFILE_METHODS
                else 0
            ),
            "query_share_with_personalized_source": (
                float(
                    np.mean(
                        list(
                            method_profile_source_used[
                                method
                            ].values()
                        )
                    )
                )
                if method in PROFILE_METHODS
                else 0.0
            ),
            "mean_dense_qchs_candidate_count": (
                float(
                    np.mean(
                        [
                            len(
                                dense_qchs_items_by_case[
                                    case_id
                                ]
                            )
                            for case_id
                            in queries["case_id"]
                        ]
                    )
                )
                if method
                in {
                    "profile_hybrid_qchs",
                    "profile_full_qchs",
                }
                else 0.0
            ),
            "mean_sparse_qchs_candidate_count": (
                float(
                    np.mean(
                        [
                            len(
                                sparse_qchs_items_by_case[
                                    case_id
                                ]
                            )
                            for case_id
                            in queries["case_id"]
                        ]
                    )
                )
                if method in PROFILE_METHODS
                else 0.0
            ),
            "mean_graph_qchs_candidate_count": (
                float(
                    np.mean(
                        [
                            len(
                                graph_qchs_items_by_case[
                                    case_id
                                ]
                            )
                            for case_id
                            in queries["case_id"]
                        ]
                    )
                )
                if method == "profile_full_qchs"
                else 0.0
            ),
        }
        for method in METHOD_ORDER
    ]
)


# =========================================================
# Final Status
# =========================================================

print("Rows: Global Review item documents", len(item_docs))
print(
    "Rows: queries with QCHS anchors",
    len(profile_case_ids),
)
print("Rows: QCHS-active cases", int(profile_diagnostics["qchs_profile_available"].sum()))
print("Rows: fallback cases", int(profile_diagnostics["profile_fallback_flag"].sum()))
print(
    "Validation: four-method personalized retrieval passed"
)

Starting sparse document tokenization
Global Review item documents: 77502


Sparse document tokenization:   0%|          | 0/77502 [00:00<?, ?it/s]

Building BM25 index
BM25 index runtime: 10.72 seconds
Starting Sparse QCHS retrieval
Queries: 2288


Sparse QCHS retrieval:   0%|          | 0/2288 [00:00<?, ?it/s]

Sparse QCHS runtime: 1594.58 seconds
Queries requiring Dense QCHS: 1513
Starting dense item embedding
Dense item documents: 77502


...

Dense QCHS runtime: 0.67 seconds
Starting Graph QCHS retrieval


Graph QCHS retrieval:   0%|          | 0/2288 [00:00<?, ?it/s]

Graph QCHS runtime: 91.65 seconds
Starting RRF fusion


RRF fusion:   0%|          | 0/2288 [00:00<?, ?it/s]

RRF fusion completed
Rows: Global Review item documents 77502
Rows: queries with QCHS anchors 1513
Rows: QCHS-active cases 1513
Rows: fallback cases 775
Validation: four-method personalized retrieval passed


In [12]:
# =========================================================
# Evaluation
# =========================================================
per_query_rows = []
candidate_list_rows = []

score_source_by_method = {
    BASELINE_METHOD_SLUG: f"notebook07_{BASELINE_METHOD_SLUG}_score",
    "profile_sparse_qchs": "notebook08_query_only_winner_plus_sparse_qchs_rrf",
    "profile_hybrid_qchs": "notebook08_query_only_winner_plus_dense_sparse_qchs_rrf",
    "profile_full_qchs": "notebook08_query_only_winner_plus_dense_sparse_graph_qchs_rrf",
}

for method in METHOD_ORDER:
    for row in queries.itertuples(index=False):
        case_id = row.case_id
        ranked_items = method_candidate_lists[method][case_id]
        candidate_scores = method_candidate_scores[method][case_id]
        target_rank = ranked_items.index(row.target_parent_asin) + 1 if row.target_parent_asin in ranked_items else 0

        baseline_items = method_candidate_lists[BASELINE_METHOD_SLUG][case_id]
        baseline_rank = baseline_items.index(row.target_parent_asin) + 1 if row.target_parent_asin in baseline_items else 0
        baseline_set = set(baseline_items)
        method_set = set(ranked_items)

        metric_row = {
            "category_id": CATEGORY_ID,
            "case_id": case_id,
            "user_id": row.user_id,
            "regime": row.regime,
            "target_parent_asin": row.target_parent_asin,
            "method_slug": method,
            "method_label": METHOD_LABELS[method],
            "baseline_method_slug": BASELINE_METHOD_SLUG,
            "baseline_method_label": BASELINE_METHOD_LABEL,
            "rank": int(target_rank),
            "baseline_rank": int(baseline_rank),
            "target_newly_entered": int(target_rank > 0 and baseline_rank == 0),
            "target_newly_lost": int(target_rank == 0 and baseline_rank > 0),
            "both_hit_rank_improvement": (
                float(baseline_rank - target_rank)
                if baseline_rank > 0 and target_rank > 0
                else np.nan
            ),
            "candidate_jaccard_vs_baseline": (
                len(baseline_set & method_set) / len(baseline_set | method_set)
                if baseline_set or method_set else 1.0
            ),
        }
        for k in EVAL_KS:
            hit, ndcg, mrr = rank_metrics(target_rank, k)
            metric_row[f"HitRate@{k}"] = hit
            metric_row[f"NDCG@{k}"] = ndcg
            metric_row[f"MRR@{k}"] = mrr
        per_query_rows.append(metric_row)

        profile = profile_by_case.get(case_id, empty_profile())
        personalized_source_used = (
            method_profile_source_used[method][case_id]
            if method in PROFILE_METHODS else False
        )
        profile_diag = profile_diagnostics_by_case.loc[case_id]
        profile_fallback_flag = bool(
            method != BASELINE_METHOD_SLUG and not personalized_source_used
        )
        profile_fallback_reason = (
            str(profile_diag["profile_fallback_reason"])
            if profile_fallback_flag else ""
        )
        passthrough = {column: getattr(row, column) for column in QUERY_PASSTHROUGH_COLUMNS}
        candidate_list_rows.append({
            "category_id": CATEGORY_ID,
            "case_id": case_id,
            "query_id": case_id,
            "user_id": row.user_id,
            "regime": row.regime,
            **passthrough,
            "target_parent_asin": row.target_parent_asin,
            "target_timestamp_ms": int(row.target_timestamp_ms),
            "prior_history_n": int(row.prior_unique_item_count),
            "query": row.active_query_text,
            "baseline_method_slug": BASELINE_METHOD_SLUG,
            "baseline_method_label": BASELINE_METHOD_LABEL,
            "method_slug": method,
            "method_label": METHOD_LABELS[method],
            "candidate_pool_role": (
                "baseline_query_only" if method == BASELINE_METHOD_SLUG else "personalized_retrieval"
            ),
            "raw_prior_item_count": int(profile_diag["raw_prior_item_count"]),
            "training_safe_prior_item_count": int(profile_diag["training_safe_prior_item_count"]),
            "qchs_selected_prior_item_count": int(profile_diag["qchs_selected_prior_item_count"]),
            "qchs_profile_safe_facet_count": int(profile_diag["qchs_profile_safe_facet_count"]),
            "qchs_brand_facet_count": int(profile_diag["qchs_brand_facet_count"]),
            "qchs_functional_facet_count": int(profile_diag["qchs_functional_facet_count"]),
            "qchs_profile_available": bool(profile_diag["qchs_profile_available"]),
            "profile_fallback_flag": int(profile_fallback_flag),
            "profile_fallback_reason": profile_fallback_reason,
            "profile_selected_prior_item_count": (
                len(profile["selected_items"]) if method != BASELINE_METHOD_SLUG else 0
            ),
            "profile_selected_facet_count": (
                len(profile["selected_facet_keys"]) if method != BASELINE_METHOD_SLUG else 0
            ),
            "profile_anchor_phrase_count": (
                len(profile["anchor_phrases"]) if method != BASELINE_METHOD_SLUG else 0
            ),
            "profile_brand_facet_count": (
                sum(str(key).startswith("brand::") for key in profile["selected_facet_keys"])
                if method != BASELINE_METHOD_SLUG else 0
            ),
            "dense_qchs_candidate_count": (
                len(dense_qchs_items_by_case[case_id])
                if method in {"profile_hybrid_qchs", "profile_full_qchs"} else 0
            ),
            "sparse_qchs_candidate_count": (
                len(sparse_qchs_items_by_case[case_id])
                if method in PROFILE_METHODS else 0
            ),
            "graph_qchs_candidate_count": (
                len(graph_qchs_items_by_case[case_id])
                if method == "profile_full_qchs" else 0
            ),
            "candidate_parent_asin_list": ranked_items,
            "candidate_brand_facet_text_list": [
                item_brand_map.get(item_id, "") for item_id in ranked_items
            ],
            "candidate_score_list": candidate_scores,
            "candidate_count": len(ranked_items),
            "candidate_score_source": (
                score_source_by_method[method]
                if method == BASELINE_METHOD_SLUG or personalized_source_used
                else score_source_by_method[BASELINE_METHOD_SLUG]
            ),
        })

per_query_metrics = pd.DataFrame(per_query_rows)
candidate_lists = pd.DataFrame(candidate_list_rows)

results_by_pool_depth = pd.DataFrame([
    {
        "category_id": CATEGORY_ID,
        "method_slug": method,
        "method_label": METHOD_LABELS[method],
        "candidate_pool_depth": k,
        "n_queries": int(len(group)),
        "HitRate": float(group[f"HitRate@{k}"].mean()),
        "NDCG": float(group[f"NDCG@{k}"].mean()),
        "MRR": float(group[f"MRR@{k}"].mean()),
    }
    for method, group in per_query_metrics.groupby("method_slug", sort=False)
    for k in EVAL_KS
])

results_by_pool_depth_regime = pd.DataFrame([
    {
        "category_id": CATEGORY_ID,
        "method_slug": method,
        "method_label": METHOD_LABELS[method],
        "regime": regime,
        "candidate_pool_depth": k,
        "n_queries": int(len(group)),
        "HitRate": float(group[f"HitRate@{k}"].mean()),
        "NDCG": float(group[f"NDCG@{k}"].mean()),
        "MRR": float(group[f"MRR@{k}"].mean()),
    }
    for (method, regime), group in per_query_metrics.groupby(["method_slug", "regime"], sort=False)
    for k in EVAL_KS
])

results_overall = results_by_pool_depth[
    results_by_pool_depth["candidate_pool_depth"].eq(MAX_RETRIEVAL_K)
].reset_index(drop=True)
results_by_regime = results_by_pool_depth_regime[
    results_by_pool_depth_regime["candidate_pool_depth"].eq(MAX_RETRIEVAL_K)
].reset_index(drop=True)

movement_summary = pd.DataFrame([
    {
        "method_slug": method,
        "method_label": METHOD_LABELS[method],
        "regime": regime,
        "n_queries": int(len(group)),
        "target_newly_entered": int(group["target_newly_entered"].sum()),
        "target_newly_lost": int(group["target_newly_lost"].sum()),
        "both_hit_rank_improvement_mean": float(group["both_hit_rank_improvement"].mean()),
        "candidate_jaccard_vs_baseline_mean": float(group["candidate_jaccard_vs_baseline"].mean()),
    }
    for (method, regime), group in per_query_metrics.groupby(["method_slug", "regime"], sort=False)
])

print("Rows: per-query metrics", len(per_query_metrics))
print("Rows: candidate lists", len(candidate_lists))


Rows: per-query metrics 9152
Rows: candidate lists 9152


In [13]:
# =========================================================
# Paired Contrasts and Personalized Winner Selection
# =========================================================
contrast_specs = {
    f"{method}_minus_query_only_winner": (method, BASELINE_METHOD_SLUG)
    for method in PROFILE_METHODS
}
metric_columns = [f"{metric}@{MAX_RETRIEVAL_K}" for metric in ["HitRate", "NDCG", "MRR"]]
wide = per_query_metrics.pivot(
    index=["case_id", "user_id", "regime"],
    columns="method_slug",
    values=metric_columns,
)

contrast_rows = []
for contrast_name, (left_method, right_method) in contrast_specs.items():
    for case_key, row in wide.iterrows():
        output = {
            "case_id": case_key[0],
            "user_id": case_key[1],
            "regime": case_key[2],
            "contrast": contrast_name,
            "left_method": left_method,
            "right_method": right_method,
        }
        for metric in metric_columns:
            output[f"delta_{metric}"] = float(row[(metric, left_method)] - row[(metric, right_method)])
        contrast_rows.append(output)
paired_contrasts = pd.DataFrame(contrast_rows)

contrast_summary_rows = []
for contrast_name, group in paired_contrasts.groupby("contrast", sort=False):
    grouped_frames = [("overall", group), *list(group.groupby("regime", sort=False))]
    for regime, regime_group in grouped_frames:
        output = {
            "contrast": contrast_name,
            "regime": regime,
            "n_queries": int(len(regime_group)),
        }
        for metric in metric_columns:
            output[f"mean_delta_{metric}"] = float(regime_group[f"delta_{metric}"].mean())
        contrast_summary_rows.append(output)
contrast_summary = pd.DataFrame(contrast_summary_rows)

runtime_summary = pd.DataFrame([
    {
        "method_slug": method,
        "method_label": METHOD_LABELS[method],
        **method_runtime_components[method],
        "online_personalized_runtime_sec": float(sum(method_runtime_components[method].values())),
        "offline_profile_sparse_index_runtime_sec": float(offline_bm25_index_runtime_sec),
        "offline_dense_index_runtime_sec": float(offline_dense_index_runtime_sec),
        "baseline_source_runtime_excluded": True,
    }
    for method in METHOD_ORDER
])

selection_metric = f"HitRate@{MAX_RETRIEVAL_K}"
selection_ndcg = f"NDCG@{MAX_RETRIEVAL_K}"
selection_mrr = f"MRR@{MAX_RETRIEVAL_K}"
profile_overall = (
    results_overall.loc[results_overall["method_slug"].isin(PROFILE_METHODS)]
    .rename(columns={"HitRate": selection_metric, "NDCG": selection_ndcg, "MRR": selection_mrr})
    .copy()
)
regime_consistency = (
    results_by_regime.loc[results_by_regime["method_slug"].isin(PROFILE_METHODS)]
    .groupby(["method_slug", "method_label"], observed=True)["HitRate"]
    .agg(
        regime_hit_rate_min="min",
        regime_hit_rate_max="max",
        regime_hit_rate_mean="mean",
        regime_hit_rate_std=lambda values: float(values.std(ddof=0)),
    )
    .reset_index()
)
personalized_method_selection = (
    profile_overall
    .merge(regime_consistency, on=["method_slug", "method_label"], how="left", validate="one_to_one")
    .merge(
        runtime_summary[["method_slug", "online_personalized_runtime_sec"]],
        on="method_slug",
        how="left",
        validate="one_to_one",
    )
)
personalized_method_selection["method_order_index"] = personalized_method_selection["method_slug"].map(
    {method: index for index, method in enumerate(PROFILE_METHODS)}
)
personalized_method_selection = personalized_method_selection.sort_values(
    [
        selection_metric,
        selection_ndcg,
        selection_mrr,
        "regime_hit_rate_min",
        "regime_hit_rate_std",
        "online_personalized_runtime_sec",
        "method_order_index",
    ],
    ascending=[False, False, False, False, True, True, True],
    kind="stable",
).reset_index(drop=True)
personalized_method_selection.insert(
    0,
    "selection_rank",
    np.arange(1, len(personalized_method_selection) + 1),
)
personalized_method_selection["primary_stage1_metric"] = selection_metric
personalized_method_selection["selection_rule"] = (
    f"maximize {selection_metric}; then {selection_ndcg}, {selection_mrr}, minimum regime HitRate, "
    "regime HitRate stability, and online personalized runtime"
)

automatic_personalized_winner = str(personalized_method_selection.iloc[0]["method_slug"])
if PERSONALIZED_WINNER_METHOD_OVERRIDE is None:
    PERSONALIZED_METHOD_SLUG = automatic_personalized_winner
    PERSONALIZED_SELECTION_SOURCE = "automatic_selection_rank_1"
else:
    PERSONALIZED_METHOD_SLUG = str(PERSONALIZED_WINNER_METHOD_OVERRIDE).strip()
    if PERSONALIZED_METHOD_SLUG not in PROFILE_METHODS:
        raise RuntimeError(
            f"PERSONALIZED_WINNER_METHOD_OVERRIDE must be one of {PROFILE_METHODS}, "
            f"found {PERSONALIZED_METHOD_SLUG!r}."
        )
    PERSONALIZED_SELECTION_SOURCE = "config_override"
PERSONALIZED_METHOD_LABEL = PROFILE_METHOD_LABELS[PERSONALIZED_METHOD_SLUG]

personalized_method_selection["is_selected_winner"] = personalized_method_selection["method_slug"].eq(
    PERSONALIZED_METHOD_SLUG
)
personalized_method_selection["selection_status"] = np.where(
    personalized_method_selection["is_selected_winner"],
    "selected_for_downstream",
    "not_selected",
)
personalized_method_selection["winner_selection_source"] = PERSONALIZED_SELECTION_SOURCE
personalized_method_selection = personalized_method_selection.drop(columns=["method_order_index"])
personalized_winner_row = personalized_method_selection.loc[
    personalized_method_selection["is_selected_winner"]
].iloc[0]

candidate_lists["selected_personalized_method"] = PERSONALIZED_METHOD_SLUG
candidate_lists["selected_personalized_method_label"] = PERSONALIZED_METHOD_LABEL
candidate_lists["is_selected_personalized_method"] = candidate_lists["method_slug"].eq(
    PERSONALIZED_METHOD_SLUG
)

print("Query-only winner:", BASELINE_METHOD_SLUG, "-", BASELINE_METHOD_LABEL)
print("Selected personalized winner:", PERSONALIZED_METHOD_SLUG, "-", PERSONALIZED_METHOD_LABEL)


Query-only winner: graph_hybrid - Graph-Hybrid
Selected personalized winner: profile_sparse_qchs - Profile Sparse QCHS


In [14]:
# =========================================================
# Validation and Export
# =========================================================
expected_methods = [BASELINE_METHOD_SLUG, *PROFILE_METHODS]
if METHOD_ORDER != expected_methods:
    raise RuntimeError(f"Notebook 08 methods must be exactly {expected_methods}.")
if set(per_query_metrics["method_slug"]) != set(expected_methods):
    raise RuntimeError("Per-query metrics do not contain the exact baseline and profile methods.")

expected_rows = len(queries) * len(METHOD_ORDER)
if len(candidate_lists) != expected_rows or len(per_query_metrics) != expected_rows:
    raise RuntimeError("Output row count must equal queries multiplied by the evaluated methods.")
if candidate_lists.duplicated(["case_id", "method_slug"]).any():
    raise RuntimeError("Candidate lists contain duplicate case-method rows.")
if not candidate_lists["candidate_count"].eq(EXPECTED_CANDIDATE_K).all():
    bad_counts = (
        candidate_lists.loc[
            ~candidate_lists["candidate_count"].eq(EXPECTED_CANDIDATE_K),
            ["case_id", "method_slug", "candidate_count"],
        ]
        .head(10)
        .to_dict("records")
    )
    raise RuntimeError(f"Candidate counts must equal exact-K={EXPECTED_CANDIDATE_K}: {bad_counts}")

expected_case_ids = set(queries["case_id"].astype(str))
for method, group in candidate_lists.groupby("method_slug", sort=False):
    if set(group["case_id"].astype(str)) != expected_case_ids:
        raise RuntimeError(f"{method} does not cover the same query IDs as the query cache.")

for row in candidate_lists.itertuples(index=False):
    items = list(row.candidate_parent_asin_list)
    brands = list(row.candidate_brand_facet_text_list)
    scores = np.asarray(row.candidate_score_list, dtype=float)
    if len(items) != row.candidate_count or len(scores) != row.candidate_count:
        raise RuntimeError("Candidate item and score lengths do not match candidate_count.")
    if len(brands) != row.candidate_count:
        raise RuntimeError("Candidate brand list length does not match candidate_count.")
    expected_brands = [item_brand_map.get(item_id, "") for item_id in items]
    if brands != expected_brands:
        raise RuntimeError("Candidate brand values do not match Notebook 04 item docs.")
    if len(items) != EXPECTED_CANDIDATE_K:
        raise RuntimeError("Candidate list order must define ranks 1 through exact-K.")
    if len(set(items)) != len(items):
        raise RuntimeError("A candidate list contains duplicate items.")
    if not set(items).issubset(item_id_set):
        raise RuntimeError("A candidate list contains an item outside the Global Review catalog.")
    if not np.isfinite(scores).all() or np.any(np.diff(scores) > 1e-12):
        raise RuntimeError("Candidate scores must be finite and non-increasing.")

for case_id in queries["case_id"]:
    if method_candidate_lists[BASELINE_METHOD_SLUG][case_id] != baseline_items_by_case[case_id]:
        raise RuntimeError("The query-only winner order changed from Notebook 07.")
    if method_candidate_scores[BASELINE_METHOD_SLUG][case_id] != baseline_scores_by_case[case_id]:
        raise RuntimeError("The query-only winner scores changed from Notebook 07.")
    for method in PROFILE_METHODS:
        if not method_profile_source_used[method][case_id]:
            if method_candidate_lists[method][case_id] != baseline_items_by_case[case_id]:
                raise RuntimeError("A profile fallback did not preserve the query-only winner order.")
            if method_candidate_scores[method][case_id] != baseline_scores_by_case[case_id]:
                raise RuntimeError("A profile fallback did not preserve the query-only winner scores.")

cold_case_ids = queries.loc[queries["regime"].eq("cold"), "case_id"]
for case_id in cold_case_ids:
    for method in PROFILE_METHODS:
        if method_candidate_lists[method][case_id] != baseline_items_by_case[case_id]:
            raise RuntimeError("A cold case did not preserve the query-only winner order.")
        if method_candidate_scores[method][case_id] != baseline_scores_by_case[case_id]:
            raise RuntimeError("A cold case did not preserve the query-only winner scores.")

non_cold_case_ids = queries.loc[queries["regime"].ne("cold"), "case_id"]
if queries.loc[queries["regime"].ne("cold"), "prior_unique_item_count"].lt(1).any():
    raise RuntimeError("Non-cold queries must have at least one training-safe prior item.")
if queries.loc[queries["regime"].eq("cold"), "stage1_usable_prior_item_count"].ne(0).any():
    raise RuntimeError("Cold queries must have zero usable Stage 1 prior items.")

candidate_regime_by_case = candidate_lists.groupby("case_id", sort=False)["regime"].first()
query_regime_by_case = queries.set_index("case_id")["regime"]
if not candidate_regime_by_case.reindex(query_regime_by_case.index).eq(query_regime_by_case).all():
    raise RuntimeError("Candidate-list regimes must match Notebook 03 regimes from the query cache.")

profile_rows_only = candidate_lists.loc[candidate_lists["method_slug"].isin(PROFILE_METHODS)].copy()
profile_rows_only["profile_fallback_flag"] = boolean_series(profile_rows_only["profile_fallback_flag"])
if (
    profile_rows_only["qchs_profile_available"].astype(bool)
    & profile_rows_only["profile_fallback_flag"]
).any():
    raise RuntimeError("A candidate-list row cannot be both QCHS-active and a fallback.")

for row in profile_rows_only.itertuples(index=False):
    personalized_source_used = method_profile_source_used[row.method_slug][row.case_id]
    if bool(row.profile_fallback_flag) == bool(personalized_source_used):
        raise RuntimeError("Profile fallback flags must be the inverse of active QCHS source use.")

fallback_case_ids = sorted(
    profile_diagnostics.loc[profile_diagnostics["profile_fallback_flag"], "case_id"].astype(str).tolist()
)
non_cold_no_alignment_fallback_case_ids = sorted(
    profile_diagnostics.loc[
        profile_diagnostics["regime"].ne("cold")
        & profile_diagnostics["profile_fallback_flag"],
        "case_id",
    ].astype(str).tolist()
)
qchs_active_case_count = int(profile_diagnostics["qchs_profile_available"].sum())
fallback_case_count = int(len(fallback_case_ids))
non_cold_fallback_case_count = int(len(non_cold_no_alignment_fallback_case_ids))
fallback_rate = float(fallback_case_count / len(queries)) if len(queries) else 0.0
fallback_counts_by_regime = (
    profile_diagnostics.loc[profile_diagnostics["profile_fallback_flag"], "regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)

if alignment_facets["is_brand"].any() or alignment_facets["is_review_derived"].any():
    raise RuntimeError("Brand or review-derived rows entered QCHS prior-item alignment.")
if profile_safe_facets["is_review_derived"].any():
    raise RuntimeError("Historical population-review facets entered the user-profile channel.")
if brand_profile_facets.empty:
    raise RuntimeError("Brand facets must be present in the personalized profile channel.")
if brand_profile_facets["is_query_safe"].any():
    raise RuntimeError("Brand profile facets must remain query-unsafe.")
if brand_profile_facets["is_product_functional_facet"].any():
    raise RuntimeError("Brand profile facets must remain separate from functional facets.")
if not brand_profile_facets["is_profile_safe"].all():
    raise RuntimeError("Brand profile facets must be profile-safe.")
if profile_safe_facets[
    [
        "is_generic_category_anchor",
        "is_generic_utility_token",
        "is_context_dependent_utility_token",
        "is_disallowed_nonfacet_source",
    ]
].any().any():
    raise RuntimeError("Generic or disallowed rows entered the profile-safe facet channel.")

if int(personalized_method_selection["is_selected_winner"].sum()) != 1:
    raise RuntimeError("Personalized method-selection table must contain exactly one selected winner.")
selected_slug = str(
    personalized_method_selection.loc[
        personalized_method_selection["is_selected_winner"], "method_slug"
    ].iloc[0]
)
if selected_slug != PERSONALIZED_METHOD_SLUG:
    raise RuntimeError("Personalized winner selection variables and table differ.")
if PERSONALIZED_METHOD_SLUG not in PROFILE_METHODS:
    raise RuntimeError("The selected personalized winner must be a personalized retrieval method.")
selected_candidate_rows = candidate_lists.loc[
    boolean_series(candidate_lists["is_selected_personalized_method"])
].copy()
if len(selected_candidate_rows) != len(queries):
    raise RuntimeError("The selected personalized method must have exactly one flagged row per query.")
if set(selected_candidate_rows["case_id"].astype(str)) != expected_case_ids:
    raise RuntimeError("Selected-personalized flags do not cover the full query set.")
if not selected_candidate_rows["method_slug"].eq(PERSONALIZED_METHOD_SLUG).all():
    raise RuntimeError("Selected-personalized flags identify a non-winner method.")
if not candidate_lists["selected_personalized_method_label"].eq(PERSONALIZED_METHOD_LABEL).all():
    raise RuntimeError("Candidate lists contain an inconsistent personalized winner label.")

forbidden_output_fragments = (
    "review_text", "review_body", "review_title", "raw_review",
    "rating", "sentiment", "prompt", "response", "llm",
)
for frame_name, frame in [
    ("candidate_lists", candidate_lists),
    ("per_query_metrics", per_query_metrics),
    ("profile_diagnostics", profile_diagnostics),
]:
    forbidden = sorted(
        column for column in frame.columns
        if any(fragment in column.lower() for fragment in forbidden_output_fragments)
    )
    if forbidden:
        raise RuntimeError(f"{frame_name} contains forbidden prior-evidence fields: {forbidden}")

print("Cases:", int(len(queries)))
print("QCHS-active cases:", qchs_active_case_count)
print("Fallback cases:", fallback_case_count)
print("Non-cold no-alignment fallbacks:", non_cold_fallback_case_count)
print("Fallback rate:", fallback_rate)
print("Fallback counts by regime:", fallback_counts_by_regime)
print("Candidate fallback validation: passed")

candidate_lists.to_parquet(CANDIDATE_LISTS_PATH, index=False)
per_query_metrics.to_parquet(PER_QUERY_METRICS_PATH, index=False)
profile_diagnostics.to_csv(PROFILE_DIAGNOSTICS_PATH, index=False, encoding="utf-8-sig")
personalized_method_selection.to_csv(METHOD_SELECTION_PATH, index=False, encoding="utf-8-sig")

results_overall.to_csv(
    OUTPUT_DIR / f"personalized_retrieval_results_overall_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)
results_by_pool_depth.to_csv(
    OUTPUT_DIR / f"personalized_retrieval_results_by_pool_depth_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)
results_by_regime.to_csv(
    OUTPUT_DIR / f"personalized_retrieval_results_by_regime_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)
results_by_pool_depth_regime.to_csv(
    OUTPUT_DIR / f"personalized_retrieval_results_by_pool_depth_regime_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)
source_contribution.to_csv(
    OUTPUT_DIR / f"personalized_retrieval_source_contribution_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)
movement_summary.to_csv(
    OUTPUT_DIR / f"personalized_retrieval_candidate_movement_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)
paired_contrasts.to_parquet(
    OUTPUT_DIR / f"personalized_retrieval_paired_contrasts_{CATEGORY_ID}.parquet",
    index=False,
)
contrast_summary.to_csv(
    OUTPUT_DIR / f"personalized_retrieval_contrast_summary_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)
runtime_summary.to_csv(
    OUTPUT_DIR / f"personalized_retrieval_runtime_top1000_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)

run_manifest = {
    "category_id": CATEGORY_ID,
    "category_label": CATEGORY_LABEL,
    "query_rows": int(len(queries)),
    "expected_regime_counts": EXPECTED_REGIME_COUNTS,
    "candidate_pool_depth": MAX_RETRIEVAL_K,
    "candidate_budget_policy": "exact_k_all_methods",
    "candidate_budget_k": MAX_RETRIEVAL_K,
    "expected_candidate_count_per_query": int(EXPECTED_CANDIDATE_K),
    "catalog_size": int(catalog_size),
    "exact_k_validation_passed": True,
    "method_order": METHOD_ORDER,
    "methods": METHOD_LABELS,
    "selected_method_slug": PERSONALIZED_METHOD_SLUG,
    "selected_method": PERSONALIZED_METHOD_LABEL,
    "selection_source": PERSONALIZED_SELECTION_SOURCE,
    "selection_status": "selected_for_downstream_candidate_export",
    "winner_contract_version": WINNER_CONTRACT_VERSION,
    "stage1_winner_contract_version": stage1_winner.get("contract_version"),
    "stage1_winner_method_key": BASELINE_METHOD_SLUG,
    "stage1_winner_method_label": BASELINE_METHOD_LABEL,
    "stage1_winner_contract_path": str(STAGE1_WINNER_MANIFEST_PATH),
    "baseline_candidate_path": str(BASELINE_CANDIDATES_PATH),
    "evidence_scope": ITEM_EVIDENCE_SCOPE,
    "query_evidence_scope": "target_review_safe_signals_only",
    "personalization_evidence_scope": PROFILE_EVIDENCE_SCOPE,
    "dense_source": DENSE_TEXT_COLUMN,
    "sparse_source": SPARSE_TEXT_COLUMN,
    "profile_graph_source": "profile_safe_metadata_functional_and_brand_facets_from_qchs_selected_prior_items",
    "user_prior_enabled": True,
    "regime_source": "strict_pre_target_history",
    "non_cold_stage1_profile_required": False,
    "non_cold_baseline_fallback_enabled": True,
    "cold_baseline_fallback_policy": "exact_query_only_winner_copy",
    "fallback_policy": "copy_baseline_candidates",
    "fallback_changes_regime": False,
    "fallback_changes_case_universe": False,
    "fallback_uses_all_prior": False,
    "fallback_is_active_personalization": False,
    "qchs_profile_coverage_rate": float(qchs_active_case_count / len(queries)) if len(queries) else 0.0,
    "qchs_active_case_count": qchs_active_case_count,
    "fallback_case_count": fallback_case_count,
    "non_cold_fallback_case_count": non_cold_fallback_case_count,
    "fallback_counts_by_regime": fallback_counts_by_regime,
    "stage1_usable_prior_item_definition": "training_safe_prior_item_with_query_alignable_metadata_facets_and_profile_safe_functional_or_brand_facets",
    "non_cold_queries_with_zero_usable_stage1_prior_items": int((
        queries["regime"].ne("cold")
        & queries["stage1_usable_prior_item_count"].eq(0)
    ).sum()),
    "raw_prior_review_text_loaded": False,
    "prior_rating_loaded": False,
    "prior_sentiment_loaded": False,
    "raw_prior_brand_column_loaded": False,
    "brand_profile_enabled": True,
    "brand_profile_source": "Notebook04 profile-safe item facets joined by prior_item_id",
    "brand_query_matching_enabled": False,
    "brand_in_synthetic_query": False,
    "historical_review_reputation_enabled": True,
    "historical_review_reputation_used_as_profile_evidence": False,
    "target_item_metadata_used_for_profile": False,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "qchs_policy": (
        "IDF-weighted query-to-prior metadata functional-facet alignment with weights 0.70 token overlap, "
        "0.20 token-boundary exact phrase, and 0.10 matched role over metadata functional facets; "
        "top-12 prior items; softmax temperature 0.25; selected items contribute all profile-safe "
        "functional and brand facets; 16 anchors with four per role"
    ),
    "profile_sparse_qchs_policy": (
        "fuse the unchanged Notebook 07 query-only winner with BM25 retrieval from the repeated active query "
        "plus QCHS anchor tokens using RRF k=60 and weights [1.0, 0.8]"
    ),
    "profile_hybrid_qchs_policy": (
        "fuse the unchanged Notebook 07 query-only winner, dense_text retrieval from the QCHS-expanded query, "
        "and sparse_text QCHS retrieval using functional-plus-brand profile anchors and RRF k=60 "
        "with weights [1.0, 0.5, 0.7]"
    ),
    "profile_full_qchs_policy": (
        "fuse the unchanged Notebook 07 query-only winner, dense QCHS, sparse QCHS, and profile-safe "
        "functional-plus-brand graph expansion using RRF k=60 and weights [1.0, 0.4, 0.6, 0.6]"
    ),
    "output_paths": {
        "candidate_lists": str(CANDIDATE_LISTS_PATH),
        "per_query_metrics": str(PER_QUERY_METRICS_PATH),
        "profile_diagnostics": str(PROFILE_DIAGNOSTICS_PATH),
        "method_selection": str(METHOD_SELECTION_PATH),
        "run_manifest": str(MANIFEST_PATH),
        "winner_contract": str(WINNER_MANIFEST_PATH),
    },
}

winner_manifest = {
    "contract_version": WINNER_CONTRACT_VERSION,
    "category_id": CATEGORY_ID,
    "category_label": CATEGORY_LABEL,
    "stage": "stage1_personalized_retrieval",
    "selection_source": PERSONALIZED_SELECTION_SOURCE,
    "automatic_winner_method_slug": automatic_personalized_winner,
    "winner_method_slug": PERSONALIZED_METHOD_SLUG,
    "winner_method_label": PERSONALIZED_METHOD_LABEL,
    "selection_rank": int(personalized_winner_row["selection_rank"]),
    "selection_metric": selection_metric,
    "selection_metrics": {
        selection_metric: float(personalized_winner_row[selection_metric]),
        selection_ndcg: float(personalized_winner_row[selection_ndcg]),
        selection_mrr: float(personalized_winner_row[selection_mrr]),
        "regime_hit_rate_min": float(personalized_winner_row["regime_hit_rate_min"]),
        "regime_hit_rate_std": float(personalized_winner_row["regime_hit_rate_std"]),
        "online_personalized_runtime_sec": float(
            personalized_winner_row["online_personalized_runtime_sec"]
        ),
    },
    "selection_rule": str(personalized_winner_row["selection_rule"]),
    "query_only_winner_method_key": BASELINE_METHOD_SLUG,
    "query_only_winner_method_label": BASELINE_METHOD_LABEL,
    "query_only_winner_candidate_path": str(BASELINE_CANDIDATES_PATH),
    "query_only_winner_contract_path": str(STAGE1_WINNER_MANIFEST_PATH),
    "candidate_lists_path": str(CANDIDATE_LISTS_PATH),
    "candidate_list_filter": {
        "column": "method_slug",
        "value": PERSONALIZED_METHOD_SLUG,
    },
    "candidate_budget_policy": "exact_k_all_methods",
    "candidate_budget_k": int(MAX_RETRIEVAL_K),
    "effective_candidate_count_per_query": int(EXPECTED_CANDIDATE_K),
    "catalog_size": int(catalog_size),
    "query_count": int(len(queries)),
    "exact_k_validation_passed": True,
    "user_prior_enabled": True,
    "regime_source": "strict_pre_target_history",
    "fallback_policy": "copy_baseline_candidates",
    "fallback_changes_regime": False,
    "fallback_changes_case_universe": False,
    "fallback_uses_all_prior": False,
    "fallback_is_active_personalization": False,
    "qchs_profile_coverage_rate": float(qchs_active_case_count / len(queries)) if len(queries) else 0.0,
    "qchs_active_case_count": qchs_active_case_count,
    "fallback_case_count": fallback_case_count,
    "non_cold_fallback_case_count": non_cold_fallback_case_count,
    "query_only_retrieval_evidence_scope": ITEM_EVIDENCE_SCOPE,
    "personalization_evidence_scope": PROFILE_EVIDENCE_SCOPE,
    "historical_review_reputation_enabled": True,
    "historical_review_reputation_used_as_profile_evidence": False,
    "brand_profile_enabled": True,
    "brand_profile_source": "Notebook04 profile-safe item facets joined by prior_item_id",
    "brand_query_matching_enabled": False,
    "brand_in_synthetic_query": False,
    "method_selection_path": str(METHOD_SELECTION_PATH),
    "personalized_run_manifest_path": str(MANIFEST_PATH),
}

with open(MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(run_manifest, file, indent=2)
with open(WINNER_MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(winner_manifest, file, indent=2)

required_outputs = [
    CANDIDATE_LISTS_PATH,
    PER_QUERY_METRICS_PATH,
    PROFILE_DIAGNOSTICS_PATH,
    METHOD_SELECTION_PATH,
    MANIFEST_PATH,
    WINNER_MANIFEST_PATH,
]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise RuntimeError(f"Missing Notebook 08 outputs: {missing_outputs}")

winner_check = load_json(WINNER_MANIFEST_PATH)
if winner_check.get("winner_method_slug") != PERSONALIZED_METHOD_SLUG:
    raise RuntimeError("Reloaded personalized winner contract method mismatch.")
if winner_check.get("query_only_winner_method_key") != BASELINE_METHOD_SLUG:
    raise RuntimeError("Reloaded personalized winner contract baseline mismatch.")
if winner_check.get("query_only_retrieval_evidence_scope") != ITEM_EVIDENCE_SCOPE:
    raise RuntimeError("Reloaded personalized winner contract evidence scope mismatch.")
if winner_check.get("historical_review_reputation_enabled") is not True:
    raise RuntimeError("Reloaded personalized winner contract must enable item-side historical review signals.")
if winner_check.get("historical_review_reputation_used_as_profile_evidence") is not False:
    raise RuntimeError("Historical review signals must not be used as QCHS profile evidence.")
if winner_check.get("brand_profile_enabled") is not True:
    raise RuntimeError("Brand must be enabled in the QCHS profile source.")
if winner_check.get("brand_query_matching_enabled") is not False:
    raise RuntimeError("Brand must not be matched from the synthetic query.")
if winner_check.get("candidate_lists_path") != str(CANDIDATE_LISTS_PATH):
    raise RuntimeError("Reloaded personalized winner contract candidate-list path mismatch.")

print("Output:", CANDIDATE_LISTS_PATH)
print("Output:", PER_QUERY_METRICS_PATH)
print("Output:", METHOD_SELECTION_PATH)
print("Output:", MANIFEST_PATH)
print("Output:", WINNER_MANIFEST_PATH)
print("Rows: candidate lists", len(candidate_lists))
print("Query-only winner:", BASELINE_METHOD_SLUG, "-", BASELINE_METHOD_LABEL)
print("Selected personalized winner:", PERSONALIZED_METHOD_SLUG, "-", PERSONALIZED_METHOD_LABEL)
print("Validation: PASS")


Cases: 2288
QCHS-active cases: 1513
Fallback cases: 775
Non-cold no-alignment fallbacks: 203
Fallback rate: 0.3387237762237762
Fallback counts by regime: {'cold': 572, 'weak': 159, 'moderate': 32, 'strong': 12}
Candidate fallback validation: passed
Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_personalized_retrieval/personalized_retrieval_candidate_lists_face.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_personalized_retrieval/personalized_retrieval_per_query_metrics_face.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_personalized_retrieval/personalized_retrieval_method_selection_face.csv
Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_personalized_retrieval/personalized_retrieval_manifest_face.json
Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_personalized_retrieval/pers

## Supplementary Diagnostic: QCHS-Active Conditional NDCG@5 Uplift

In [15]:
# =========================================================
# QCHS-active conditional NDCG@5 uplift
# =========================================================
from pathlib import Path
import json
import numpy as np
import pandas as pd

METRIC = "NDCG@5"
ZERO_TOL = 1e-12


def read_json_required(path, name):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def normalize_slug(value):
    return str(value).strip() if value is not None else ""


required_config_objects = [
    "CATEGORY_ID", "OUTPUT_DIR", "REGIME_ORDER", "CANDIDATE_LISTS_PATH",
    "PER_QUERY_METRICS_PATH", "PROFILE_DIAGNOSTICS_PATH", "MANIFEST_PATH",
    "WINNER_MANIFEST_PATH",
]
missing_config_objects = [name for name in required_config_objects if name not in globals()]
if missing_config_objects:
    raise RuntimeError(
        "Run the Notebook 08 setup/config cells first so category-local output paths are defined. "
        f"Missing: {missing_config_objects}"
    )

OUTPUT_DIR = Path(OUTPUT_DIR)
CANDIDATE_LISTS_PATH = Path(CANDIDATE_LISTS_PATH)
PER_QUERY_METRICS_PATH = Path(PER_QUERY_METRICS_PATH)
PROFILE_DIAGNOSTICS_PATH = Path(PROFILE_DIAGNOSTICS_PATH)
MANIFEST_PATH = Path(MANIFEST_PATH)
WINNER_MANIFEST_PATH = Path(WINNER_MANIFEST_PATH)

notebook08_manifest = read_json_required(MANIFEST_PATH, "Notebook 08 run manifest")
notebook08_winner = read_json_required(WINNER_MANIFEST_PATH, "Notebook 08 winner manifest")
if notebook08_manifest.get("category_id") != CATEGORY_ID:
    raise RuntimeError("Notebook 08 run manifest category_id mismatch.")
if notebook08_winner.get("category_id") != CATEGORY_ID:
    raise RuntimeError("Notebook 08 winner manifest category_id mismatch.")

manifest_output_paths = notebook08_manifest.get("output_paths", {})
expected_output_paths = {
    "candidate_lists": CANDIDATE_LISTS_PATH,
    "per_query_metrics": PER_QUERY_METRICS_PATH,
    "profile_diagnostics": PROFILE_DIAGNOSTICS_PATH,
    "run_manifest": MANIFEST_PATH,
    "winner_contract": WINNER_MANIFEST_PATH,
}
for key, expected_path in expected_output_paths.items():
    observed_text = manifest_output_paths.get(key)
    if not observed_text or Path(observed_text) != expected_path:
        raise RuntimeError(f"Notebook 08 manifest output path mismatch for {key}: {observed_text}")
if Path(notebook08_winner.get("candidate_lists_path", "")) != CANDIDATE_LISTS_PATH:
    raise RuntimeError("Notebook 08 winner manifest candidate_lists_path mismatch.")

if "BASELINE_METHOD_SLUG" not in globals():
    BASELINE_METHOD_SLUG = normalize_slug(
        notebook08_manifest.get("stage1_winner_method_key")
        or notebook08_winner.get("query_only_winner_method_key")
    )
if "PERSONALIZED_METHOD_SLUG" not in globals():
    PERSONALIZED_METHOD_SLUG = normalize_slug(
        notebook08_manifest.get("selected_method_slug")
        or notebook08_winner.get("winner_method_slug")
    )
P0_METHOD = normalize_slug(BASELINE_METHOD_SLUG)
P1_METHOD = normalize_slug(PERSONALIZED_METHOD_SLUG)
if not P0_METHOD or not P1_METHOD:
    raise RuntimeError("Notebook 08 manifests do not identify the baseline and personalized method slugs.")
if normalize_slug(notebook08_manifest.get("stage1_winner_method_key")) != P0_METHOD:
    raise RuntimeError("Notebook 08 manifest baseline method slug differs from the active contract.")
if normalize_slug(notebook08_manifest.get("selected_method_slug")) != P1_METHOD:
    raise RuntimeError("Notebook 08 manifest personalized method slug differs from the active contract.")
if normalize_slug(notebook08_winner.get("winner_method_slug")) != P1_METHOD:
    raise RuntimeError("Notebook 08 winner manifest personalized method slug differs from the active contract.")

missing_output_frames = [
    name for name in ["per_query_metrics", "candidate_lists", "profile_diagnostics"]
    if name not in globals()
]
if missing_output_frames:
    missing_paths = [
        str(path) for path in [CANDIDATE_LISTS_PATH, PER_QUERY_METRICS_PATH, PROFILE_DIAGNOSTICS_PATH]
        if not path.exists()
    ]
    if missing_paths:
        raise FileNotFoundError(
            "Notebook 08 canonical outputs are required when dataframes are not already in memory. "
            f"Missing files: {missing_paths}"
        )
    per_query_metrics = pd.read_parquet(PER_QUERY_METRICS_PATH)
    candidate_lists = pd.read_parquet(CANDIDATE_LISTS_PATH)
    profile_diagnostics = pd.read_csv(PROFILE_DIAGNOSTICS_PATH)

ACTIVE_CASE_PATH = OUTPUT_DIR / f"qchs_active_case_metrics_{CATEGORY_ID}.parquet"
ACTIVE_SUMMARY_PATH = OUTPUT_DIR / f"qchs_active_conditional_summary_{CATEGORY_ID}.csv"
COVERAGE_PATH = OUTPUT_DIR / f"qchs_coverage_summary_{CATEGORY_ID}.csv"
FALLBACK_QC_PATH = OUTPUT_DIR / f"qchs_fallback_identity_qc_{CATEGORY_ID}.csv"


def require_columns(frame, columns, name):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise RuntimeError(f"{name} is missing columns: {missing}")


def as_bool(values):
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(values):
        return values.fillna(0).astype(float).ne(0)
    return values.fillna("").astype(str).str.strip().str.lower().isin(
        {"1", "true", "t", "yes", "y"}
    )


def as_list(value):
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if hasattr(value, "tolist"):
        value = value.tolist()
        return value if isinstance(value, list) else [value]
    raise TypeError(f"Expected a list-like value, found {type(value)!r}")


def scopes(frame):
    yield "overall", "overall", frame
    yield "non_cold", "non_cold", frame.loc[frame["regime"].ne("cold")]
    observed = frame["regime"].dropna().astype(str).unique().tolist()
    for regime in dict.fromkeys([*list(REGIME_ORDER), *observed]):
        yield "regime", str(regime), frame.loc[frame["regime"].eq(regime)]


# 1. Pre-outcome QCHS-active definition.
profile_required = [
    "case_id", "user_id", "regime", "target_parent_asin",
    "qchs_selected_prior_item_count", "qchs_profile_safe_facet_count",
    "qchs_profile_available", "profile_fallback_flag",
]
require_columns(profile_diagnostics, profile_required, "profile_diagnostics")
profile = profile_diagnostics.copy()
for column in ["case_id", "user_id", "regime", "target_parent_asin"]:
    profile[column] = profile[column].astype(str)
if profile["case_id"].duplicated().any():
    raise RuntimeError("profile_diagnostics must contain one row per case_id.")

profile["qchs_selected_prior_item_count"] = pd.to_numeric(
    profile["qchs_selected_prior_item_count"], errors="raise"
).astype(int)
profile["qchs_profile_safe_facet_count"] = pd.to_numeric(
    profile["qchs_profile_safe_facet_count"], errors="raise"
).astype(int)

# Notebook 08 profile usability contract: at least one selected prior item and at least one profile-safe facet.
profile["qchs_active"] = (
    profile["qchs_selected_prior_item_count"].gt(0)
    & profile["qchs_profile_safe_facet_count"].gt(0)
)
if not profile["qchs_active"].eq(as_bool(profile["qchs_profile_available"])).all():
    raise RuntimeError("Recomputed qchs_active disagrees with qchs_profile_available.")
if not (~profile["qchs_active"]).eq(as_bool(profile["profile_fallback_flag"])).all():
    raise RuntimeError("profile_fallback_flag must be the inverse of qchs_active.")
if profile.loc[profile["regime"].eq("cold"), "qchs_active"].any():
    raise RuntimeError("Cold cases must not be QCHS-active.")


# 2. Pair P1-only winner and P0 on the exact same cases.
metric_required = [
    "case_id", "user_id", "regime", "target_parent_asin", "method_slug", "rank", METRIC,
]
require_columns(per_query_metrics, metric_required, "per_query_metrics")
metrics = per_query_metrics.copy()
for column in ["case_id", "user_id", "regime", "target_parent_asin", "method_slug"]:
    metrics[column] = metrics[column].astype(str)
keys = ["case_id", "user_id", "regime", "target_parent_asin"]
p0 = metrics.loc[metrics["method_slug"].eq(P0_METHOD), [*keys, "rank", METRIC]].rename(
    columns={"rank": "p0_rank", METRIC: "p0_ndcg_at_5"}
)
p1 = metrics.loc[metrics["method_slug"].eq(P1_METHOD), [*keys, "rank", METRIC]].rename(
    columns={"rank": "p1_rank", METRIC: "p1_ndcg_at_5"}
)
if p0["case_id"].duplicated().any() or p1["case_id"].duplicated().any():
    raise RuntimeError("P0 and P1 must each contain one metric row per case_id.")
paired = p0.merge(p1, on=keys, validate="one_to_one")
if set(paired["case_id"]) != set(profile["case_id"]):
    raise RuntimeError("Paired P0/P1 metrics do not cover the profile case universe.")

profile_optional = [
    "prior_review_event_count", "prior_unique_item_count", "raw_prior_item_count",
    "training_safe_prior_item_count", "prior_depth_bin", "stage1_usable_prior_item_count",
    "stage1_profile_safe_prior_item_count", "qchs_selected_ratio", "qchs_alignment_mean",
    "qchs_alignment_max", "qchs_attention_entropy", "qchs_brand_facet_count",
    "qchs_functional_facet_count", "qchs_anchor_phrase_count", "profile_fallback_reason",
]
profile_columns = [
    *keys, "qchs_selected_prior_item_count", "qchs_profile_safe_facet_count", "qchs_active",
    *[column for column in profile_optional if column in profile.columns],
]
paired = paired.merge(profile[profile_columns], on=keys, validate="one_to_one")
paired["p0_rank"] = pd.to_numeric(paired["p0_rank"], errors="raise").astype(int)
paired["p1_rank"] = pd.to_numeric(paired["p1_rank"], errors="raise").astype(int)
paired["p0_ndcg_at_5"] = pd.to_numeric(paired["p0_ndcg_at_5"], errors="raise")
paired["p1_ndcg_at_5"] = pd.to_numeric(paired["p1_ndcg_at_5"], errors="raise")
paired["delta_ndcg_at_5"] = paired["p1_ndcg_at_5"] - paired["p0_ndcg_at_5"]
paired["target_newly_entered_pool"] = paired["p0_rank"].eq(0) & paired["p1_rank"].gt(0)
paired["target_newly_lost_pool"] = paired["p0_rank"].gt(0) & paired["p1_rank"].eq(0)


# 3. Confirm fallback identity; candidate change is descriptive, never a selection rule.
candidate_required = [
    "case_id", "method_slug", "candidate_parent_asin_list", "candidate_score_list",
    "profile_fallback_flag",
]
require_columns(candidate_lists, candidate_required, "candidate_lists")
candidates = candidate_lists.copy()
candidates["case_id"] = candidates["case_id"].astype(str)
candidates["method_slug"] = candidates["method_slug"].astype(str)
c0 = candidates.loc[candidates["method_slug"].eq(P0_METHOD), [
    "case_id", "candidate_parent_asin_list", "candidate_score_list"
]].rename(columns={"candidate_parent_asin_list": "p0_items", "candidate_score_list": "p0_scores"})
c1 = candidates.loc[candidates["method_slug"].eq(P1_METHOD), [
    "case_id", "candidate_parent_asin_list", "candidate_score_list", "profile_fallback_flag"
]].rename(columns={
    "candidate_parent_asin_list": "p1_items", "candidate_score_list": "p1_scores",
    "profile_fallback_flag": "p1_fallback",
})
candidate_qc = c0.merge(c1, on="case_id", validate="one_to_one").merge(
    profile[["case_id", "regime", "qchs_active"]], on="case_id", validate="one_to_one"
)
if set(candidate_qc["case_id"]) != set(profile["case_id"]):
    raise RuntimeError("P0/P1 candidate rows do not cover the profile case universe.")
if not as_bool(candidate_qc["p1_fallback"]).eq(~candidate_qc["qchs_active"]).all():
    raise RuntimeError("Selected P1 fallback flags disagree with qchs_active.")

candidate_qc["candidate_id_order_identical"] = [
    as_list(left) == as_list(right) for left, right in zip(candidate_qc["p0_items"], candidate_qc["p1_items"])
]
candidate_qc["candidate_scores_identical"] = [
    bool(
        np.asarray(as_list(left), dtype=float).shape == np.asarray(as_list(right), dtype=float).shape
        and np.allclose(
            np.asarray(as_list(left), dtype=float), np.asarray(as_list(right), dtype=float),
            rtol=0.0, atol=0.0, equal_nan=True,
        )
    )
    for left, right in zip(candidate_qc["p0_scores"], candidate_qc["p1_scores"])
]
candidate_qc["candidate_list_changed"] = ~candidate_qc["candidate_id_order_identical"]
paired = paired.merge(
    candidate_qc[["case_id", "candidate_id_order_identical", "candidate_scores_identical", "candidate_list_changed"]],
    on="case_id", validate="one_to_one",
)

fallback_qc_rows = []
for scope_type, scope_value, group in scopes(candidate_qc):
    fallback = group.loc[~group["qchs_active"]]
    identity_pass = bool(
        fallback["candidate_id_order_identical"].all()
        and fallback["candidate_scores_identical"].all()
    ) if len(fallback) else True
    fallback_qc_rows.append({
        "category_id": CATEGORY_ID, "scope_type": scope_type, "scope_value": scope_value,
        "fallback_case_count": int(len(fallback)),
        "candidate_id_order_identity_count": int(fallback["candidate_id_order_identical"].sum()),
        "candidate_score_identity_count": int(fallback["candidate_scores_identical"].sum()),
        "fallback_identity_pass": identity_pass,
    })
qchs_fallback_identity_qc = pd.DataFrame(fallback_qc_rows)
if not qchs_fallback_identity_qc["fallback_identity_pass"].all():
    qchs_fallback_identity_qc.to_csv(FALLBACK_QC_PATH, index=False, encoding="utf-8-sig")
    raise RuntimeError("Fallback identity failed; QC was exported. Do not use the active-only result yet.")


# 4. Export coverage and active-only NDCG@5 uplift, with ITT reconciliation.
profile_qc = profile.merge(
    candidate_qc[["case_id", "candidate_list_changed"]], on="case_id", validate="one_to_one"
)
coverage_rows, summary_rows = [], []
for scope_type, scope_value, group in scopes(profile_qc):
    active = group.loc[group["qchs_active"]]
    n_total, n_active = len(group), len(active)
    coverage_rows.append({
        "category_id": CATEGORY_ID, "baseline_method_slug": P0_METHOD,
        "personalized_method_slug": P1_METHOD, "scope_type": scope_type,
        "scope_value": scope_value, "total_case_count": int(n_total),
        "qchs_active_case_count": int(n_active), "fallback_case_count": int(n_total - n_active),
        "qchs_active_rate": float(n_active / n_total) if n_total else np.nan,
        "fallback_rate": float((n_total - n_active) / n_total) if n_total else np.nan,
        "active_candidate_list_changed_count": int(active["candidate_list_changed"].sum()),
        "active_candidate_list_changed_rate": float(active["candidate_list_changed"].mean()) if n_active else np.nan,
        "activation_definition": "selected_prior_count>0 and usable_profile_available",
    })

for scope_type, scope_value, all_cases in scopes(paired):
    active = all_cases.loc[all_cases["qchs_active"]]
    n_total, n_active = len(all_cases), len(active)
    active_rate = float(n_active / n_total) if n_total else np.nan
    delta = active["delta_ndcg_at_5"].to_numpy(float)
    full_delta = float(all_cases["delta_ndcg_at_5"].mean()) if n_total else np.nan
    active_delta = float(delta.mean()) if n_active else np.nan
    weighted_delta = float(active_rate * active_delta) if n_active else (0.0 if n_total else np.nan)
    positive = int((delta > ZERO_TOL).sum())
    negative = int((delta < -ZERO_TOL).sum())
    tie = int(n_active - positive - negative)
    summary_rows.append({
        "category_id": CATEGORY_ID, "baseline_method_slug": P0_METHOD,
        "personalized_method_slug": P1_METHOD, "scope_type": scope_type,
        "scope_value": scope_value, "metric": METRIC, "total_case_count": int(n_total),
        "qchs_active_case_count": int(n_active), "qchs_active_rate": active_rate,
        "active_baseline_ndcg_at_5": float(active["p0_ndcg_at_5"].mean()) if n_active else np.nan,
        "active_personalized_ndcg_at_5": float(active["p1_ndcg_at_5"].mean()) if n_active else np.nan,
        "active_mean_delta_ndcg_at_5": active_delta,
        "active_median_delta_ndcg_at_5": float(np.median(delta)) if n_active else np.nan,
        "active_positive_delta_count": positive, "active_tie_count": tie,
        "active_negative_delta_count": negative,
        "active_positive_delta_rate": float(positive / n_active) if n_active else np.nan,
        "active_tie_rate": float(tie / n_active) if n_active else np.nan,
        "active_negative_delta_rate": float(negative / n_active) if n_active else np.nan,
        "active_candidate_list_changed_count": int(active["candidate_list_changed"].sum()),
        "active_candidate_list_changed_rate": float(active["candidate_list_changed"].mean()) if n_active else np.nan,
        "active_target_newly_entered_pool_count": int(active["target_newly_entered_pool"].sum()),
        "active_target_newly_lost_pool_count": int(active["target_newly_lost_pool"].sum()),
        "canonical_full_sample_mean_delta_ndcg_at_5": full_delta,
        "coverage_weighted_active_mean_delta_ndcg_at_5": weighted_delta,
        "itt_reconciliation_absolute_error": abs(full_delta - weighted_delta) if n_total else np.nan,
        "estimand_label": "conditional_descriptive_uplift_among_pre_outcome_qchs_active_cases",
    })

qchs_coverage_summary = pd.DataFrame(coverage_rows)
qchs_active_conditional_summary = pd.DataFrame(summary_rows)
max_error = float(qchs_active_conditional_summary["itt_reconciliation_absolute_error"].fillna(0).max())
if max_error > ZERO_TOL:
    raise RuntimeError(f"ITT reconciliation failed; maximum error={max_error}")

qchs_active_case_metrics = paired.loc[paired["qchs_active"]].copy()
qchs_active_case_metrics.insert(0, "category_id", CATEGORY_ID)
qchs_active_case_metrics.insert(1, "baseline_method_slug", P0_METHOD)
qchs_active_case_metrics.insert(2, "personalized_method_slug", P1_METHOD)
qchs_active_case_metrics["activation_definition"] = (
    "qchs_selected_prior_item_count > 0 and qchs_profile_safe_facet_count > 0"
)
qchs_active_case_metrics["selection_uses_candidate_outcome"] = False

qchs_active_case_metrics.to_parquet(ACTIVE_CASE_PATH, index=False)
qchs_active_conditional_summary.to_csv(ACTIVE_SUMMARY_PATH, index=False, encoding="utf-8-sig")
qchs_coverage_summary.to_csv(COVERAGE_PATH, index=False, encoding="utf-8-sig")
qchs_fallback_identity_qc.to_csv(FALLBACK_QC_PATH, index=False, encoding="utf-8-sig")

print("Canonical outputs modified: False")
print("P0 / P1:", P0_METHOD, "/", P1_METHOD)
print("Total / active / fallback:", len(profile), len(qchs_active_case_metrics), int((~profile["qchs_active"]).sum()))
print("Maximum ITT reconciliation error:", max_error)
for path in [ACTIVE_CASE_PATH, ACTIVE_SUMMARY_PATH, COVERAGE_PATH, FALLBACK_QC_PATH]:
    print("Output:", path)
display(qchs_coverage_summary)
display(qchs_active_conditional_summary)
display(qchs_fallback_identity_qc)


Canonical outputs modified: False
P0 / P1: graph_hybrid / profile_sparse_qchs
Total / active / fallback: 2288 1513 775
Maximum ITT reconciliation error: 2.710505431213761e-20
Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_personalized_retrieval/qchs_active_case_metrics_face.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_personalized_retrieval/qchs_active_conditional_summary_face.csv
Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_personalized_retrieval/qchs_coverage_summary_face.csv
Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_personalized_retrieval/qchs_fallback_identity_qc_face.csv


,category_id,baseline_method_slug,personalized_method_slug,scope_type,scope_value,total_case_count,qchs_active_case_count,fallback_case_count,qchs_active_rate,fallback_rate,active_candidate_list_changed_count,active_candidate_list_changed_rate,activation_definition
0,face,graph_hybrid,profile_sparse_qchs,overall,overall,2288,1513,775,0.661276,0.338724,1513,1.0,selected_prior_count>0 and usable_profile_avai...
1,face,graph_hybrid,profile_sparse_qchs,non_cold,non_cold,1716,1513,203,0.881702,0.118298,1513,1.0,selected_prior_count>0 and usable_profile_avai...
2,face,graph_hybrid,profile_sparse_qchs,regime,cold,572,0,572,0.000000,1.000000,0,NaN,selected_prior_count>0 and usable_profile_avai...
3,face,graph_hybrid,profile_sparse_qchs,regime,weak,572,413,159,0.722028,0.277972,413,1.0,selected_prior_count>0 and usable_profile_avai...
4,face,graph_hybrid,profile_sparse_qchs,regime,moderate,572,540,32,0.944056,0.055944,540,1.0,selected_prior_count>0 and usable_profile_avai...
5,face,graph_hybrid,profile_sparse_qchs,regime,strong,572,560,12,0.979021,0.020979,560,1.0,selected_prior_count>0 and usable_profile_avai...


,category_id,baseline_method_slug,personalized_method_slug,scope_type,scope_value,metric,total_case_count,qchs_active_case_count,qchs_active_rate,active_baseline_ndcg_at_5,...,active_tie_rate,active_negative_delta_rate,active_candidate_list_changed_count,active_candidate_list_changed_rate,active_target_newly_entered_pool_count,active_target_newly_lost_pool_count,canonical_full_sample_mean_delta_ndcg_at_5,coverage_weighted_active_mean_delta_ndcg_at_5,itt_reconciliation_absolute_error,estimand_label
0,face,graph_hybrid,profile_sparse_qchs,overall,overall,NDCG@5,2288,1513,0.661276,0.011848,...,0.984137,0.005948,1513,1.0,80,35,0.001613,0.001613,0.000000e+00,conditional_descriptive_uplift_among_pre_outco...
1,face,graph_hybrid,profile_sparse_qchs,non_cold,non_cold,NDCG@5,1716,1513,0.881702,0.011848,...,0.984137,0.005948,1513,1.0,80,35,0.002151,0.002151,0.000000e+00,conditional_descriptive_uplift_among_pre_outco...
2,face,graph_hybrid,profile_sparse_qchs,regime,cold,NDCG@5,572,0,0.000000,NaN,...,NaN,NaN,0,NaN,0,0,0.000000,0.000000,0.000000e+00,conditional_descriptive_uplift_among_pre_outco...
3,face,graph_hybrid,profile_sparse_qchs,regime,weak,NDCG@5,572,413,0.722028,0.004675,...,0.987893,0.004843,413,1.0,37,16,-0.000121,-0.000121,2.710505e-20,conditional_descriptive_uplift_among_pre_outco...
4,face,graph_hybrid,profile_sparse_qchs,regime,moderate,NDCG@5,572,540,0.944056,0.007947,...,0.987037,0.003704,540,1.0,27,9,0.003529,0.003529,0.000000e+00,conditional_descriptive_uplift_among_pre_outco...
5,face,graph_hybrid,profile_sparse_qchs,regime,strong,NDCG@5,572,560,0.979021,0.020901,...,0.978571,0.008929,560,1.0,16,10,0.003046,0.003046,0.000000e+00,conditional_descriptive_uplift_among_pre_outco...


,category_id,scope_type,scope_value,fallback_case_count,candidate_id_order_identity_count,candidate_score_identity_count,fallback_identity_pass
0,face,overall,overall,775,775,775,True
1,face,non_cold,non_cold,203,203,203,True
2,face,regime,cold,572,572,572,True
3,face,regime,weak,159,159,159,True
4,face,regime,moderate,32,32,32,True
5,face,regime,strong,12,12,12,True
